# Hyperliquid

Download data from hyperliquid (https://hyperfoundation.org/). Hyperliquid is a DEX.

In [ ]:
#| default_exp hyperliquid

In [ ]:
#|hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
#from nbdev.showdoc import *
import json
#from typing import List, Dict, Tuple, Optional, Union, Any, Callable
from hyperliquid.utils import constants
import os
import time
import requests as _requests

import eth_account
from eth_account.signers.local import LocalAccount
from hyperliquid.exchange import Exchange
from hyperliquid.info import Info
import pandas as pd
from datetime import datetime
import datetime as dt
import numpy as np

## Connect to Hyperliquid API 
### setup function

In [ ]:
#| export
def setup(base_url=None, skip_ws=False, perp_dexs=None,config='../config_hyperliquid.json',debug=False):
    # This function is copied from hyperliquid-python-sdk/examples/example_utils.py
    # for setting up the environment in our script.
    # config_path = os.path.join(os.path.dirname(__file__), "config.json")
    config_path = config
    with open(config_path) as f:
        config = json.load(f)
    account: LocalAccount = eth_account.Account.from_key(config["secret_key"])
    address = config["account_address"]
    if address == "":
        address = account.address
        if debug:
            print("Running with account address:", address)
    if address != account.address:
        if debug:
            print("Running with agent address:", account.address)
    info = Info(base_url, skip_ws, perp_dexs=perp_dexs)
    user_state = info.user_state(address)
    spot_user_state = info.spot_user_state(address)
    margin_summary = user_state["marginSummary"]
    if float(margin_summary["accountValue"]) == 0 and len(spot_user_state["balances"]) == 0:
        print("Not running the example because the provided account has no equity.")
        url = info.base_url.split(".", 1)[1]
        error_string = f"No accountValue:\nIf you think this is a mistake, make sure that {address} has a balance on {url}.\nIf address shown is your API wallet address, update the config to specify the address of your account, not the address of the API wallet."
        raise Exception(error_string)
    exchange = Exchange(account, base_url, account_address=address, perp_dexs=perp_dexs)
    return address, info, exchange

### Example

In [ ]:
#| eval:false
address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)

## HIP-3 / RWA dex support
### Helper functions for resolving builder-dex tickers (e.g. XLE → xyz:XLE)

Hyperliquid hosts third-party perpetual markets on "builder dexes" (HIP-3).
These tickers use a `dex:COIN` naming convention (e.g. `xyz:XLE`).
The helpers below let every data-download function transparently resolve
a plain ticker like `"XLE"` to `"xyz:XLE"` and fall back to a raw HTTP
request when the Python SDK's internal name map doesn't know the coin.

In [ ]:
#| export
_BASE_URL = "https://api.hyperliquid.xyz"
_HIP3_PREFIXES = ["xyz", "flx", "vntl", "hyna", "km", "abcd", "cash", "para", "mkts"]
_resolve_cache = {}

def _hl_post(payload, base_url=None, timeout=30):
    """Low-level POST to Hyperliquid /info endpoint (no SDK needed)."""
    url = (base_url or _BASE_URL) + "/info"
    resp = _requests.post(url, json=payload,
                          headers={"Content-Type": "application/json"},
                          timeout=timeout)
    resp.raise_for_status()
    return resp.json()

def hyperliquid_perp_dexs(base_url=None):
    """Return list of perp dex name strings ('' = native, 'xyz', etc.)."""
    data = _hl_post({"type": "perpDexs"}, base_url=base_url)
    return [d.get("name", "") if d else "" for d in data]

def resolve_hyperliquid_ticker(coin, info=None, base_url=None):
    """Resolve a plain ticker to its dex-prefixed form if needed.

    - If *coin* already contains ':' it is returned as-is.
    - If *coin* exists in the native perp universe it is returned as-is.
    - Otherwise each HIP-3 dex universe is searched for ``{dex}:{coin}``.
    - Results are cached so repeated calls are cheap.

    Args:
        coin (str): Ticker such as ``"ETH"`` or ``"XLE"``.
        info (Info, optional): SDK Info client (used for native universe check).
        base_url (str, optional): API base URL override.

    Returns:
        str: Resolved ticker (e.g. ``"xyz:XLE"``).  Falls back to *coin*
             unchanged if nothing is found.
    """
    if ":" in coin:
        return coin
    if coin in _resolve_cache:
        return _resolve_cache[coin]

    # Check native universe first
    try:
        bu = base_url or _BASE_URL
        native_meta = _hl_post({"type": "meta"}, base_url=bu)
        native_names = {u["name"] for u in native_meta.get("universe", [])}
        if coin in native_names:
            _resolve_cache[coin] = coin
            return coin
    except Exception:
        pass

    # Search HIP-3 dexes
    for prefix in _HIP3_PREFIXES:
        try:
            dex_meta = _hl_post({"type": "meta", "dex": prefix}, base_url=bu)
            dex_names = {u["name"] for u in dex_meta.get("universe", [])}
            candidate = f"{prefix}:{coin}"
            if candidate in dex_names:
                _resolve_cache[coin] = candidate
                return candidate
        except Exception:
            continue

    _resolve_cache[coin] = coin
    return coin

def _ticker_file_safe(ticker):
    """Convert a dex-prefixed ticker to a filesystem-safe name (`:` → `-`)."""
    return ticker.replace(":", "-")

def _ticker_from_file_safe(name):
    """Reverse of ``_ticker_file_safe`` — first hyphen becomes `:`."""
    parts = name.split("-", 1)
    if len(parts) == 2 and parts[0] in _HIP3_PREFIXES:
        return f"{parts[0]}:{parts[1]}"
    return name

## Get historical prerpetual price data
### retrive_hyperliquid_perp_price function

In [ ]:
#| export
def retrieve_hyperliquid_perp_price(coin="ETH", interval="1h", 
                                end_date=datetime.now(dt.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
                                start_date=(datetime.now(dt.timezone.utc)-pd.Timedelta(days=2)).strftime('%Y-%m-%dT%H:%M:%SZ'),
                                info=None):
    """
    Retrieves historical candle data from Hyperliquid for a given coin and time interval.

    Args:
        coin (str, optional): Coin symbol (e.g. "ETH", "XLE"). Defaults to "ETH".
            Plain RWA tickers are auto-resolved (e.g. "XLE" → "xyz:XLE").
        interval (str, optional): Candle interval ("1m", "5m", "15m", "1h", "4h", "1d"). Defaults to "1h".
        end_date (str, optional): End datetime in ISO 8601 format. Defaults to current UTC time.
        start_date (str, optional): Start datetime in ISO 8601 format. Defaults to 2 days before end_date.
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.

    Returns:
        pandas.DataFrame: DataFrame containing the OHLCV data with columns:
            - datetime: Timestamp for the candle (UTC)
            - open: Opening price of the interval
            - high: Highest traded price in the interval
            - low: Lowest traded price in the interval
            - close: Closing price of the interval
            - volume: Trading volume in the interval
            - coin: Coin symbol
        Returns None if the API request fails or returns no data.

    Notes:
        - All datetime values are in UTC timezone
        - Requires Hyperliquid Info client to be initialized
    """
    user_coin = coin
    if info is None:
        from hyperliquid.info import Info
        from hyperliquid.utils import constants
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)

    # Resolve RWA / HIP-3 ticker
    resolved = resolve_hyperliquid_ticker(coin, info=info)

    try:
        # Convert datetime strings to Unix milliseconds timestamps
        start_dt = pd.to_datetime(start_date)
        end_dt = pd.to_datetime(end_date)
        
        start_time_ms = int(start_dt.timestamp() * 1000)
        end_time_ms = int(end_dt.timestamp() * 1000)
        
        # Try SDK first, fall back to raw API for HIP-3 coins
        candles = None
        try:
            candles = info.candles_snapshot(name=resolved, interval=interval, 
                                           startTime=start_time_ms, endTime=end_time_ms)
        except KeyError:
            # SDK doesn't know this coin – use raw HTTP request
            payload = {
                "type": "candleSnapshot",
                "req": {
                    "coin": resolved,
                    "interval": interval,
                    "startTime": start_time_ms,
                    "endTime": end_time_ms,
                }
            }
            base_url = getattr(info, 'base_url', _BASE_URL)
            candles = _hl_post(payload, base_url=base_url)
        
        if not candles:
            return None
        
        # Convert to DataFrame
        df = pd.DataFrame(candles)
        
        # Convert timestamp to datetime
        df['datetime'] = pd.to_datetime(df['t'], unit='ms')
        
        # Rename columns to match coinbase format
        df = df.rename(columns={
            'o': 'open',
            'h': 'high', 
            'l': 'low',
            'c': 'close',
            'v': 'volume'
        })
        
        # Add coin column (use the name the caller passed)
        df['coin'] = user_coin
        
        # Sort by datetime and reorder columns
        df = df.sort_values(by='datetime')
        df = df[['datetime', 'open', 'high', 'low', 'close', 'volume', 'coin']]
        df = df.astype({'open': 'float64', 'high': 'float64', 'low': 'float64', 'close': 'float64', 'volume': 'float64'})
        
        return df.reset_index(drop=True)
        
    except Exception as e:
        print(f"Error retrieving candles for {coin}: {e}")
        return None

### Example

You will need a api key and secret key from the Hyperliquid API. Store this into the `config_hyperliquid.json` file in the same directory as your script. See: https://app.hyperliquid.xyz/API

The best is to first call the `setup` function once to initialize the Hyperliquid Info client. Then pass "info" to any function that requires it. That will avoid calling the `setup` function multiple times.

Typical usage:

In [ ]:
#| eval: false
perp = retrieve_hyperliquid_perp_price(coin="ETH", interval="1h",info=info)
print(perp.head())

             datetime    open    high     low   close     volume coin
0 2026-07-24 23:00:00  1858.8  1861.1  1858.1  1861.0  2792.8168  ETH
1 2026-07-25 00:00:00  1860.7  1863.7  1856.3  1859.3  4137.4891  ETH
2 2026-07-25 01:00:00  1859.3  1862.6  1857.4  1859.3  4557.0876  ETH
3 2026-07-25 02:00:00  1859.3  1860.7  1857.3  1858.3  4404.2636  ETH
4 2026-07-25 03:00:00  1858.3  1859.5  1854.8  1858.3  4298.4131  ETH


## List of spot tickers
### spot_tickers function


In [ ]:
#| export
def spot_tickers(coin="ETH", base='USDC',info=None):
    """
    Retrieves current tickers for a given coin.

    Args:
        coin (str, optional): Coin symbol (e.g. "ETH"). Defaults to "ETH".
        base (str, optional): Coin symbol (e.g. "USDC"). Defaults to "USDC".
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.

    Returns:
        spot ticker (non intuitive symbol)
        Returns None if the API request fails or returns no data.

    Notes:
        - Requires Hyperliquid Info client to be initialized
    """
    if info is None:
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    maps =info.spot_meta_and_asset_ctxs()
    # change of ticker for ETH and BTC.... they should have U in front of the name.... Hyperliquid's naming convention...
    if coin.upper() == "ETH":
        coin = "UETH"
    elif coin.upper() == "BTC":
        coin = "UBTC"
    elif coin.upper() == "DOGE":
        coin = "UDOGE"
    elif coin.upper() == "SOL":
        coin = "USOL"
    if base.upper() == "USDC":
        base = "USDC"
    # Find the index of the coin and base in the universe of tokens
    # If not found, return None
    id_coin, id_base = None, None
    for token_ctx in maps[0]['tokens']:
        if token_ctx['name'] == coin:
            id_coin = token_ctx['index']
        elif token_ctx['name'] == base:
            id_base = token_ctx['index']
    if id_coin is None or id_base is None:
        return None
    for i in maps[0]['universe']:
        if i['tokens'] == [id_coin,id_base]:
            return i['name']
    id_coin, id_base = None, None
    for token_ctx in maps[0]['tokens']:
        if token_ctx['name'] == coin:
            id_coin = token_ctx['index']
        elif token_ctx['name'] == base:
            id_base = token_ctx['index']
    if id_coin is None or id_base is None:
        return None
    for i in maps['universe']:
        if i['tokens'] == [id_coin,id_base]:
            return i['name']
    return None                

### Example

In [ ]:
#| eval: false
stk = spot_tickers(info=info,coin="ETH",base="USDC")
print(f'spot ticker for ETH: ', stk)

spot ticker for ETH:  @151


## Get historical spot price
### retrieve_hyperliquid_spot_price function

In [ ]:
#| export
def retrieve_hyperliquid_spot_price(coin="ETH", base='USDC',interval="1h", 
                                end_date=datetime.now(dt.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
                                start_date=(datetime.now(dt.timezone.utc)-pd.Timedelta(days=2)).strftime('%Y-%m-%dT%H:%M:%SZ'),
                                info=None,recheck=False):
    # get the ticker for the given coin
    if recheck:
        ticker = spot_tickers(coin=coin,base=base,info=info)
    else:
        spot_dict = {'BTC': '@142',
                        'ETH': '@151',
                        'SOL': '@156',
                        'HYPE': '@107',
                        'TRUMP': '@9',
                        'BERA': '@117',
                        'PUMP': '@20'}
        ticker = spot_dict.get(coin.upper(), None)
        if ticker is None:
            ticker = spot_tickers(coin=coin,base=base,info=info)

    if ticker is None:
        print(f"{coin} is not listed.")
        return pd.DataFrame()
    # get the price for the given ticker... same function used for perpetuals but
    # ticker price is different...
    try:
        price = retrieve_hyperliquid_perp_price(coin=ticker, interval=interval, 
                                end_date=end_date,
                                start_date=start_date,
                                info=info)
        price['coin'] = coin
        return price
    except Exception as e:
        print(f"Error retrieving price for {coin}: {e}")
        return None

### Example

The ticker for Ethereum (ETH) is "@151" ("UETH"). There are other unusual choices but the function handles these choices by changing the ticker inside the function. 

In [ ]:
#| eval: false
spot=retrieve_hyperliquid_spot_price(info=info)
print(spot.tail())

              datetime    open    high     low   close    volume coin
44 2026-07-26 19:00:00  1913.0  1913.8  1910.2  1912.5  336.0437  ETH
45 2026-07-26 20:00:00  1912.5  1912.9  1910.2  1912.1  136.4614  ETH
46 2026-07-26 21:00:00  1912.8  1927.2  1912.2  1923.9  139.3486  ETH
47 2026-07-26 22:00:00  1924.4  1951.0  1924.4  1948.5  461.1441  ETH
48 2026-07-26 23:00:00  1951.1  1964.7  1948.5  1959.5  142.2609  ETH


If you want to find out which ticker is used for a given coin, you can use the `spot_tickers` function. For example, for Ethereum (ETH) quoted in USDC is:

In [ ]:
#| eval: false
stk = spot_tickers(info=info,coin="ETH",base="USDC")
print(f'spot ticker for ETH: ', stk)

spot ticker for ETH:  @151


## List all tokens in Hyperliquid
### hyperliquid_tokens function

In [ ]:
#| export
def hyperliquid_tokens(info=None,rm_delisted=True,dex=None):
    """
    List perpetual tokens available on Hyperliquid.

    Args:
        info (Info, optional): SDK Info client. Created automatically if *None*.
        rm_delisted (bool): Remove delisted and isolated-only tokens (default *True*).
        dex (str, optional): Which perp dex universe to query.

            - ``None`` (default) – native perps only (unchanged legacy behaviour).
            - A dex name string (e.g. ``"xyz"``) – that builder-dex universe.
              Returned ``name`` values are already prefixed (e.g. ``"xyz:XLE"``).
            - ``"all"`` – concatenates native **and** every HIP-3 dex universe.

    Returns:
        pandas.DataFrame with columns ``szDecimals``, ``name``, ``maxLeverage``, etc.
    """
    if info is None:
        from hyperliquid.info import Info
        from hyperliquid.utils import constants
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)

    base_url = getattr(info, 'base_url', _BASE_URL)

    def _fetch_universe(dex_name):
        meta = _hl_post({"type": "meta", "dex": dex_name} if dex_name else {"type": "meta"},
                        base_url=base_url)
        tokens = meta.get('universe', [])
        df = pd.DataFrame(tokens)
        if df.empty:
            return df
        df['isDelisted'] = ~df['isDelisted'].isna()
        df['onlyIsolated'] = ~df['onlyIsolated'].isna()
        if rm_delisted:
            df = df.loc[(~df['isDelisted']) & (~df['onlyIsolated'])]
        return df

    if dex is None:
        # Legacy path – native universe via SDK
        tokens = info.meta_and_asset_ctxs()[0].get('universe')
        df = pd.DataFrame(tokens)
        df['isDelisted'] = ~df['isDelisted'].isna()
        df['onlyIsolated'] = ~df['onlyIsolated'].isna()
        if rm_delisted:
            df = df.loc[(~df['isDelisted']) & (~df['onlyIsolated'])]
        return df
    elif dex == "all":
        frames = [_fetch_universe(None)]
        for prefix in _HIP3_PREFIXES:
            try:
                f = _fetch_universe(prefix)
                if not f.empty:
                    frames.append(f)
            except Exception:
                continue
        return pd.concat(frames, ignore_index=True)
    else:
        return _fetch_universe(dex)

### Example

In [ ]:
#| eval: false
tokens = hyperliquid_tokens(info)
print(tokens)

     szDecimals   name  maxLeverage  marginTableId  isDelisted  onlyIsolated  \
0             5    BTC           40             56       False         False   
1             4    ETH           25             55       False         False   
2             2   ATOM            5              5       False         False   
4             1   DYDX            5              5       False         False   
5             2    SOL           20             54       False         False   
..          ...    ...          ...            ...         ...           ...   
226           2   DASH            5              5       False         False   
227           0    SKR            3              3       False         False   
228           0  AZTEC            3              3       False         False   
229           0   CHIP            3              3       False         False   
230           0   GRAM            5              5       False         False   

    marginMode  
0          NaN  
1    

## Get funding rate history
### retrieve_hyperliquid_funding_history function

In [ ]:
#| export
def funding_calc(rate,premium,max_rate=0.0005,min_rate=-0.0005):
    return premium+max(min(rate-premium, max_rate), min_rate)

In [ ]:
#| export
def retrieve_hyperliquid_funding_history(coin="ETH", 
                                        end_date=datetime.now(dt.timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
                                        start_date=(datetime.now(dt.timezone.utc)-pd.Timedelta(days=2)).strftime('%Y-%m-%dT%H:%M:%SZ'),
                                        info=None,
                                        calc=False):
    """
    Retrieves funding rate history from Hyperliquid for a given coin and time period.

    Args:
        coin (str, optional): Coin symbol (e.g. "ETH", "XLE"). Defaults to "ETH".
            Plain RWA tickers are auto-resolved (e.g. "XLE" → "xyz:XLE").
        end_date (str, optional): End datetime in ISO 8601 format. Defaults to current UTC time.
        start_date (str, optional): Start datetime in ISO 8601 format. Defaults to 7 days before end_date.
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.

    Returns:
        pandas.DataFrame: DataFrame containing the funding history with columns:
            - datetime: Timestamp for the funding rate (UTC)
            - funding_rate: The funding rate value
            - premium: The premium component
            - coin: Coin symbol
        Returns None if the API request fails or returns no data.

    Notes:
        - All datetime values are in UTC timezone
        - Funding rates are typically updated every hour
        - Requires Hyperliquid Info client to be initialized
    """
    user_coin = coin
    if info is None:
        from hyperliquid.info import Info
        from hyperliquid.utils import constants
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    
    # Resolve RWA / HIP-3 ticker
    resolved = resolve_hyperliquid_ticker(coin, info=info)

    try:
        # Convert datetime strings to Unix milliseconds timestamps
        start_dt = pd.to_datetime(start_date)
        end_dt = pd.to_datetime(end_date)
        
        start_time_ms = int(start_dt.timestamp() * 1000)
        end_time_ms = int(end_dt.timestamp() * 1000)
        
        # Try SDK first, fall back to raw API for HIP-3 coins
        funding_rates = None
        try:
            funding_rates = info.funding_history(
                name=resolved,
                startTime=start_time_ms,
                endTime=end_time_ms
            )
        except KeyError:
            # SDK doesn't know this coin – use raw HTTP request
            payload = {
                "type": "fundingHistory",
                "coin": resolved,
                "startTime": start_time_ms,
                "endTime": end_time_ms,
            }
            base_url = getattr(info, 'base_url', _BASE_URL)
            funding_rates = _hl_post(payload, base_url=base_url)
        
        if not funding_rates:
            return None
        
        # Convert to DataFrame
        df = pd.DataFrame(funding_rates)
        
        # Convert timestamp from milliseconds to datetime
        df['datetime'] = pd.to_datetime(df['time'], unit='ms')
        
        # Rename columns for clarity
        df = df.rename(columns={
            'fundingRate': 'funding_rate',
            'premium': 'premium'
        })
        
        # Convert multiple columns to float
        df[['funding_rate', 'premium']] = df[['funding_rate', 'premium']].astype(float)
        # Add coin column (use the name the caller passed)
        df['coin'] = user_coin
        
        # Sort by datetime and reorder columns
        df = df.sort_values(by='datetime')
        df = df[['datetime', 'funding_rate', 'premium', 'coin']]
        
        # Drop the original time column if it exists
        if 'time' in df.columns:
            df = df.drop(columns=['time'])
        
        df['fund_calc'] = df['funding_rate']
        if calc:
            vectorized_funding_calc = np.vectorize(funding_calc)
            df['fund_calc'] = vectorized_funding_calc(df['funding_rate'], df['premium'])
        
        return df.reset_index(drop=True)
        
    except Exception as e:
        print(f"Error retrieving funding history for {coin}: {e}")
        return None

### Example

In [ ]:
#| eval: false
f_r = retrieve_hyperliquid_funding_history(info=info,calc=True)
print(f_r.tail())

                  datetime  funding_rate   premium coin  fund_calc
43 2026-07-26 19:00:00.031      0.000013 -0.000234  ETH   0.000013
44 2026-07-26 20:00:00.057      0.000013 -0.000285  ETH   0.000012
45 2026-07-26 21:00:00.001      0.000013 -0.000335  ETH   0.000012
46 2026-07-26 22:00:00.023      0.000013 -0.000301  ETH   0.000012
47 2026-07-26 23:00:00.005      0.000013 -0.000202  ETH   0.000013


## Unified function for easy data retrieval
### retrieve_hyperliquid_data function

In [ ]:
#| export
def retrieve_hyperliquid_data(ticker="ETH", 
                              data_type="perp",
                              start_date=None,
                              end_date=None,
                              lookback=2,
                              interval="1h",
                              base="USDC",
                              round_to_hour=False,
                              info=None):
    """
    Unified function to retrieve funding rates, perpetual prices, or spot prices from Hyperliquid.
    
    Args:
        ticker (str, optional): Coin symbol (e.g. "ETH", "BTC"). Defaults to "ETH".
        data_type (str, optional): Type of data to retrieve - "funding", "perp", or "spot". Defaults to "perp".
        start_date (str, optional): Start date as string. Can be:
            - ISO format: "2024-01-15T10:30:00Z" or "2024-01-15T10:30:00"
            - Date only: "2024-01-15"
            - If None, calculated from lookback. Defaults to None.
        end_date (str, optional): End date as string (same formats as start_date).
            - If None, uses current UTC time. Defaults to None.
        lookback (int, optional): Number of days to look back from end_date if start_date is None. 
            Defaults to 2.
        interval (str, optional): Candle interval for perp/spot data ("1m", "5m", "15m", "1h", "4h", "1d"). 
            Defaults to "1h". Not used for funding rates.
        base (str, optional): Base currency for spot prices (e.g. "USDC"). Defaults to "USDC".
            Not used for perp or funding rates.
        round_to_hour (bool, optional): If True, rounds start_date and end_date to nearest hour.
            Useful for funding rates which update hourly. Defaults to False.
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
    
    Returns:
        pandas.DataFrame: DataFrame containing the requested data with appropriate columns:
            - For "funding": datetime, funding_rate, premium, coin
            - For "perp": datetime, open, high, low, close, volume, coin
            - For "spot": datetime, open, high, low, close, volume, coin
        Returns None if the API request fails or returns no data.
    
    Examples:
        # Get 7 days of funding rates for ETH, rounded to hour
        df = retrieve_hyperliquid_data("ETH", "funding", lookback=7, round_to_hour=True, info=info)
        
        # Get perp prices between specific dates with 4h interval
        df = retrieve_hyperliquid_data("BTC", "perp", 
                                      start_date="2024-01-01", 
                                      end_date="2024-01-15",
                                      interval="4h", info=info)
        
        # Get spot prices for last 30 days with 1h interval
        df = retrieve_hyperliquid_data("ETH", "spot", lookback=30, 
                                      interval="1h", base="USDC", info=info)
    
    Notes:
        - All datetime values are in UTC timezone
        - Valid data_type values: "funding", "perp", "spot"
        - Funding rates are updated hourly, so round_to_hour=True is recommended
        - Requires Hyperliquid Info client to be initialized
    """
    # Validate data_type
    valid_types = ["funding", "perp", "spot"]
    if data_type not in valid_types:
        raise ValueError(f"data_type must be one of {valid_types}, got '{data_type}'")
    
    # Initialize info client if not provided
    if info is None:
        from hyperliquid.info import Info
        from hyperliquid.utils import constants
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    
    # Handle end_date
    if end_date is None:
        end_dt = pd.Timestamp.now(tz='UTC')
    else:
        # Parse end_date string
        try:
            end_dt = pd.to_datetime(end_date)
        except Exception as e:
            print(f"Error parsing end_date '{end_date}': {e}")
            return None

    # Handle start_date
    if start_date is None:
        # Calculate from lookback
        start_dt = end_dt - pd.Timedelta(days=lookback)
    else:
        # Parse start_date string
        try:
            start_dt = pd.to_datetime(start_date)
        except Exception as e:
            print(f"Error parsing start_date '{start_date}': {e}")
            return None
    
    # Convert to ISO format strings
    start_date_str = start_dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    end_date_str = end_dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    
    # Call appropriate function based on data_type
    try:
        if data_type == "funding":
            df = retrieve_hyperliquid_funding_history(
                coin=ticker,
                start_date=start_date_str,
                end_date=end_date_str,
                info=info
                )
            if round_to_hour:
                df['datetime'] = df['datetime'].apply(lambda x: x.round('h'))
            return df
        elif data_type == "perp":
            return retrieve_hyperliquid_perp_price(
                coin=ticker,
                interval=interval,
                start_date=start_date_str,
                end_date=end_date_str,
                info=info
            )
        elif data_type == "spot":
            return retrieve_hyperliquid_spot_price(
                coin=ticker,
                base=base,
                interval=interval,
                start_date=start_date_str,
                end_date=end_date_str,
                info=info
            )
    except Exception as e:
        print(f"Error retrieving {data_type} data for {ticker}: {e}")
        return None

### Example

In [ ]:
#| eval: false
## Example 1: Get funding rates for last 7 days, rounded to hour
funding_df = retrieve_hyperliquid_data(
    ticker="ETH",
    data_type="funding",
    lookback=7,
    round_to_hour=True,
    info=info
)
print(funding_df.tail())

               datetime  funding_rate   premium coin  fund_calc
163 2026-07-26 19:00:00      0.000013 -0.000234  ETH   0.000013
164 2026-07-26 20:00:00      0.000013 -0.000285  ETH   0.000013
165 2026-07-26 21:00:00      0.000013 -0.000335  ETH   0.000013
166 2026-07-26 22:00:00      0.000013 -0.000301  ETH   0.000013
167 2026-07-26 23:00:00      0.000013 -0.000202  ETH   0.000013


In [ ]:
#| eval: false
## Example 2: Get perpetual prices with specific dates and 4h interval
perp_df = retrieve_hyperliquid_data(
    ticker="BTC",
    data_type="perp",
    start_date="2026-01-01",
    end_date="2026-01-15",
    interval="4h",
    info=info
)
print(perp_df.tail())

              datetime     open     high      low    close       volume coin
80 2026-01-14 08:00:00  95182.0  95302.0  94714.0  94773.0   2307.95490  BTC
81 2026-01-14 12:00:00  94773.0  97102.0  94647.0  96737.0  13517.53853  BTC
82 2026-01-14 16:00:00  96737.0  97765.0  96263.0  97241.0  11110.15613  BTC
83 2026-01-14 20:00:00  97241.0  97949.0  96799.0  96917.0   6714.43893  BTC
84 2026-01-15 00:00:00  96917.0  96975.0  95747.0  95970.0   5284.50121  BTC


In [ ]:
#| eval: false
## Example 3: Get spot prices for last 30 days
spot_df = retrieve_hyperliquid_data(
    ticker="ETH",
    data_type="spot",
    lookback=30,
    interval="1h",
    base="USDC",
    info=info
)
print(spot_df.tail())


               datetime    open    high     low   close    volume coin
716 2026-07-26 19:00:00  1913.0  1913.8  1910.2  1912.5  336.0437  ETH
717 2026-07-26 20:00:00  1912.5  1912.9  1910.2  1912.1  136.4614  ETH
718 2026-07-26 21:00:00  1912.8  1927.2  1912.2  1923.9  139.3486  ETH
719 2026-07-26 22:00:00  1924.4  1951.0  1924.4  1948.5  461.1441  ETH
720 2026-07-26 23:00:00  1951.1  1964.7  1948.5  1959.5  142.2609  ETH


In [ ]:
#| eval: false
## Example 4: Using datetime strings with time information
data_df = retrieve_hyperliquid_data(
    ticker="ETH",
    data_type="perp",
    lookback=2,
    interval="1h",
    info=info
)
print(data_df.tail())

              datetime    open    high     low   close      volume coin
44 2026-07-26 19:00:00  1914.0  1915.4  1911.5  1913.6   2512.5447  ETH
45 2026-07-26 20:00:00  1913.6  1914.0  1910.9  1912.8   2440.3697  ETH
46 2026-07-26 21:00:00  1912.7  1929.9  1912.7  1925.3  13779.9050  ETH
47 2026-07-26 22:00:00  1925.3  1951.1  1925.3  1949.0  47844.9203  ETH
48 2026-07-26 23:00:00  1949.0  1967.9  1949.0  1960.5  20710.9213  ETH


In [ ]:
#| eval: false
## Example 5: Get funding rates with date-only strings
funding_df = retrieve_hyperliquid_data(
    ticker="BTC",
    data_type="funding",
    start_date="2025-08-29",
    end_date="2025-09-01",
    round_to_hour=True,
    info=info
)
print(funding_df)

              datetime  funding_rate   premium coin  fund_calc
0  2025-08-29 00:00:00      0.000013  0.000251  BTC   0.000013
1  2025-08-29 01:00:00      0.000013  0.000234  BTC   0.000013
2  2025-08-29 02:00:00      0.000013  0.000124  BTC   0.000013
3  2025-08-29 03:00:00      0.000013  0.000068  BTC   0.000013
4  2025-08-29 04:00:00      0.000013  0.000086  BTC   0.000013
..                 ...           ...       ...  ...        ...
67 2025-08-31 19:00:00      0.000013 -0.000035  BTC   0.000013
68 2025-08-31 20:00:00      0.000013 -0.000082  BTC   0.000013
69 2025-08-31 21:00:00      0.000013 -0.000028  BTC   0.000013
70 2025-08-31 22:00:00      0.000013 -0.000059  BTC   0.000013
71 2025-08-31 23:00:00      0.000013 -0.000012  BTC   0.000013

[72 rows x 5 columns]


In [ ]:
#| eval: false
data_df = retrieve_hyperliquid_data(
    ticker="ETH",
    data_type="perp",
    start_date="2026-06-15T10:30:00",
    end_date="2026-06-20T15:45:00",
    interval="1h",
    info=info
)
data_df.tail()

,datetime,open,high,low,close,volume,coin
121,2026-06-20 11:00:00,1727.5,1731.1,1725.9,1727.1,2295.1586,ETH
122,2026-06-20 12:00:00,1727.1,1730.5,1723.6,1726.7,20751.8753,ETH
123,2026-06-20 13:00:00,1726.6,1726.6,1712.7,1717.3,18955.0975,ETH
124,2026-06-20 14:00:00,1717.4,1745.5,1708.0,1733.1,21337.8294,ETH
125,2026-06-20 15:00:00,1733.2,1750.0,1729.3,1739.6,46985.6872,ETH


## Order book data
### retrieve_hyperliquid_l2_snapshot

The will return the snapshot of the order book for the specified ticker in the moment you call the function. It does not provide historical data. 

This is good only for perpetual markets.

In [ ]:
def unix_to_datetime(t):
    return datetime.fromtimestamp(t/1000).strftime("%Y-%m-%d %H:%M:%S.%f")
#The funding rates are reset every hour.
#t = datetime.datetime.fromtimestamp(i['time']/1000).strftime("%Y-%m-%d %H:%M:%S.%f")
#print(t)

In [ ]:

#| export
def retrieve_hyperliquid_l2_snapshot(coin="ETH", info=None):
    """
    Retrieves current L2 order book snapshot from Hyperliquid for a given coin.
    
    Args:
        coin (str, optional): Coin symbol (e.g. "ETH", "BTC", "XLE"). Defaults to "ETH".
            Plain RWA tickers are auto-resolved (e.g. "XLE" → "xyz:XLE").
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
    
    Returns:
        pandas.DataFrame: DataFrame containing the order book snapshot with columns:
            - datetime: Timestamp of the snapshot (UTC)
            - side: Order side ("bid" or "ask")
            - price: Price level
            - size: Total size at this price level
            - num_orders: Number of orders at this price level
        Returns None if the API request fails or returns no data.
    
    Notes:
        - This is a snapshot at the moment the function is called
        - All datetime values are in UTC timezone
        - Bids are sorted from highest to lowest price
        - Asks are sorted from lowest to highest price
        - Requires Hyperliquid Info client to be initialized
    """
    if info is None:
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    
    # Resolve RWA / HIP-3 ticker
    resolved = resolve_hyperliquid_ticker(coin, info=info)

    try:
        # Try SDK first, fall back to raw API for HIP-3 coins
        l2_data = None
        try:
            l2_data = info.l2_snapshot(name=resolved)
        except KeyError:
            payload = {"type": "l2Book", "coin": resolved}
            base_url = getattr(info, 'base_url', _BASE_URL)
            l2_data = _hl_post(payload, base_url=base_url)
        
        if not l2_data or 'levels' not in l2_data:
            return None
        
        # Convert timestamp to datetime
        timestamp = pd.to_datetime(l2_data['time'], unit='ms')
        
        # Extract bid and ask levels
        bids = l2_data['levels'][0]  # First element is bids
        asks = l2_data['levels'][1]  # Second element is asks
        
        # Create list to store all rows
        rows = []
        
        # Process bids
        for bid in bids:
            rows.append({
                'datetime': timestamp,
                'side': 'bid',
                'price': float(bid['px']),
                'size': float(bid['sz']),
                'num_orders': int(bid['n'])
            })
        
        # Process asks
        for ask in asks:
            rows.append({
                'datetime': timestamp,
                'side': 'ask',
                'price': float(ask['px']),
                'size': float(ask['sz']),
                'num_orders': int(ask['n'])
            })
        
        # Create DataFrame
        df = pd.DataFrame(rows)
        
        # Reorder columns for consistency
        df = df[['datetime', 'side', 'price', 'size', 'num_orders']]
        
        return df
        
    except Exception as e:
        print(f"Error retrieving L2 snapshot for {coin}: {e}")
        return None

### Example

In [ ]:
#| eval: false

# Example usage: Get L2 order book snapshot for ETH
l2_snapshot = retrieve_hyperliquid_l2_snapshot(coin="ETH", info=info)
print(l2_snapshot.head(5))
print(l2_snapshot.tail(5))

# Check the structure
print(f"\nTotal levels: {len(l2_snapshot)}")
print(f"Bids: {len(l2_snapshot[l2_snapshot['side'] == 'bid'])}")
print(f"Asks: {len(l2_snapshot[l2_snapshot['side'] == 'ask'])}")
print(f"Snapshot time: {l2_snapshot['datetime'].iloc[0]}")

                 datetime side   price     size  num_orders
0 2026-07-26 23:16:41.579  bid  1960.5   4.3721           2
1 2026-07-26 23:16:41.579  bid  1960.4  34.3144           4
2 2026-07-26 23:16:41.579  bid  1960.3   1.4367           1
3 2026-07-26 23:16:41.579  bid  1960.2  16.8822           4
4 2026-07-26 23:16:41.579  bid  1960.1  32.5682          10
                  datetime side   price      size  num_orders
35 2026-07-26 23:16:41.579  ask  1962.1  202.2812          11
36 2026-07-26 23:16:41.579  ask  1962.2  224.6602          15
37 2026-07-26 23:16:41.579  ask  1962.3  231.3283          11
38 2026-07-26 23:16:41.579  ask  1962.4  198.4616          13
39 2026-07-26 23:16:41.579  ask  1962.5  265.0086          15

Total levels: 40
Bids: 20
Asks: 20
Snapshot time: 2026-07-26 23:16:41.579000


## Mid prices
### hyperliquid_mids

In [ ]:
#| export
def hyperliquid_mids(coin=None,info=None,typecast_to_float=True):
    """
    Retrieves current mid prices from Hyperliquid for specified coins or all available coins.
    
    Args:
        coin (str, optional): Coin symbol (e.g. "ETH", "BTC", "XLE").
            Plain RWA tickers are auto-resolved (e.g. "XLE" → "xyz:XLE").
            For HIP-3/RWA coins not in the native allMids endpoint the mid
            is computed from the L2 book.
            If None, retrieves mids for all available coins. Defaults to None.
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
    
    Returns:
        dictionary: Dictionary containing mid prices with keys as coin symbols and values as mid prices
        Returns None if the API request fails or returns no data.
    
    Examples:
        # Get mid prices for specific coins
        dic = hyperliquid_mids(info=info)
    
    Notes:
        - This is a snapshot at the moment the function is called
        - Mid price is calculated as (best_bid + best_ask) / 2
        - Requires Hyperliquid Info client to be initialized
    """
    if info is None:
        address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
    
    try:
        # Get all mid prices from Hyperliquid
        mids_data = info.all_mids()
        
        if not mids_data:
            return None
        if coin:
            # Try direct lookup first
            if coin in mids_data.keys():
                return float(mids_data[coin])
            # Try resolving RWA / HIP-3 ticker
            resolved = resolve_hyperliquid_ticker(coin, info=info)
            if resolved in mids_data.keys():
                return float(mids_data[resolved])
            # HIP-3 coins may not appear in allMids – compute from L2 book
            if resolved != coin or ":" in coin:
                try:
                    l2 = retrieve_hyperliquid_l2_snapshot(coin=coin, info=info)
                    if l2 is not None and not l2.empty:
                        best_bid = l2.loc[l2['side'] == 'bid', 'price'].max()
                        best_ask = l2.loc[l2['side'] == 'ask', 'price'].min()
                        return float((best_bid + best_ask) / 2)
                except Exception:
                    pass
            return None
            
        if typecast_to_float: # turn off to save time...
            for i in mids_data.keys():
                mids_data[i] = float(mids_data[i])

        return mids_data
        
    except Exception as e:
        print(f"Error retrieving mid prices: {e}")
        return None


### Example

In [ ]:
#| eval: false
hyperliquid_mids(coin="ETH",info=info)

1960.55

In [ ]:
#| eval: false
hyperliquid_mids(info=info)

{'#5090': 0.5,
 '#5091': 0.5,
 '#5100': 0.783925,
 '#5101': 0.216075,
 '#5110': 0.00553,
 '#5111': 0.99447,
 '#5120': 0.184345,
 '#5121': 0.815655,
 '#9290': 0.980635,
 '#9291': 0.019365,
 '#9300': 0.982445,
 '#9301': 0.017555,
 '#9310': 0.986455,
 '#9311': 0.013545,
 '#9320': 0.894995,
 '#9321': 0.105005,
 '#9330': 0.5,
 '#9331': 0.5,
 '#9340': 0.020495,
 '#9341': 0.979505,
 '#9350': 0.62101,
 '#9351': 0.37899,
 '#9360': 0.07873,
 '#9361': 0.92127,
 '0G': 0.17671,
 '2Z': 0.061502,
 '@1': 18.227,
 '@10': 7.355e-05,
 '@100': 0.003556,
 '@101': 0.126125,
 '@102': 0.00769,
 '@103': 5.393e-05,
 '@104': 0.027789,
 '@105': 0.1419,
 '@106': 0.005081,
 '@107': 59.6785,
 '@108': 0.028835,
 '@109': 0.00048383,
 '@11': 0.000405,
 '@110': 0.01703,
 '@111': 0.026029,
 '@112': 0.0005408,
 '@113': 0.0002112,
 '@114': 0.0001255,
 '@115': 0.039845,
 '@116': 1.17e-05,
 '@117': 0.0013545,
 '@118': 0.015663,
 '@119': 0.009494,
 '@12': 4.494e-05,
 '@120': 0.016848,
 '@121': 0.0092,
 '@122': 0.003342,
 '@12

## Data management

Similar to coinbase.py, this script saves the data in a specified format. These functions handles duplicates and sorts by date. For hourly data, it aligns to the hour.

### HyperliquidDataManager base class

In [ ]:

#| export
def save_hyperliquid_file(df, folder_path, file_name, type="parquet"):
    """
    Save a pandas DataFrame to a file in either CSV or Parquet format.

    Args:
        df (pandas.DataFrame): The DataFrame to save
        folder_path (str): Directory path where the file will be saved
        file_name (str): Name of the file without extension
        type (str, optional): File format - either "csv" or "parquet". Defaults to "parquet"

    The function saves the DataFrame to the specified path, handling the file extension automatically.
    For CSV files, the index is not saved. For Parquet files, default Parquet settings are used.
    Creates the folder if it doesn't exist.
    """
    # Create folder if it doesn't exist
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    
    if type == "csv":
        df.to_csv(f"{folder_path}/{file_name}.csv", index=False)
    elif type == "parquet":
        df.to_parquet(f"{folder_path}/{file_name}.parquet")
    else:
        raise ValueError(f"Type {type} not supported. Use 'csv' or 'parquet'")

In [ ]:

#| export
class HyperliquidDataManager:
    """
    Base class for managing Hyperliquid data files.
    
    Handles reading, updating, and saving data for different data types (perp, spot, funding).
    Data is stored in organized folders by data type, with files named by token and interval.
    
    Args:
        ticker (str or list, optional): Token symbol(s) to manage. If None, uses all available tokens.
        data_dir (str, optional): Base directory for data storage. Defaults to "../data/hyperliquid"
        interval (str, optional): Time interval for data ("1m", "5m", "15m", "1h", "4h", "1d"). 
            Defaults to "1h". Not used for funding data.
        file_type (str, optional): File format - "parquet" or "csv". Defaults to "parquet"
        update (bool, optional): If True, checks for and downloads new data. Defaults to False
        save (bool, optional): If True, saves updated data back to file. Defaults to False
        refresh_hours (int, optional): Hours of data to refresh when updating. Defaults to 24
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
        verbose (bool, optional): If True, prints progress messages. Defaults to True
    
    Attributes:
        data_type (str): Type of data managed by this instance ("perp", "spot", "funding")
        data (dict): Dictionary mapping tickers to their DataFrames
    """
    
    def __init__(self, ticker=None, data_dir="../data/hyperliquid", interval="1h",
                 file_type="parquet", update=False, save=False, refresh_hours=24,
                 info=None, verbose=True,data_type=None):
        self.ticker = ticker
        self.data_dir = data_dir
        self.interval = interval
        self.file_type = file_type
        self.update = update
        self.save = save
        self.refresh_hours = refresh_hours
        self.info = info
        self.verbose = verbose
        self.data = {}
        self.data_type = data_type  # To be set by derived classes
        
        # Initialize info client if needed
        if self.info is None:
            if self.update:
                raise ValueError("To update data, Info client must be provided")
        
        # Get list of tickers to process
        self._initialize_tickers()
        
        # Create directory structure if needed
        self._ensure_directories()
    
    def _initialize_tickers(self):
        """Initialize the list of tickers to process."""
        if self.ticker is None:
            # Get all available tokens
            self.tickers = self._get_all_tickers()
        elif isinstance(self.ticker, str):
            self.tickers = [self.ticker]
        elif isinstance(self.ticker, list):
            self.tickers = self.ticker
        else:
            raise ValueError("ticker must be None, str, or list")
    
    def _get_all_tickers(self):
        """Get all available tickers for this data type. To be implemented by derived classes."""
        raise NotImplementedError("Derived classes must implement _get_all_tickers")
    
    def _ensure_directories(self):
        """Create directory structure if it doesn't exist."""
        if self.data_type is None:
            raise ValueError("data_type must be set by derived class")
        
        full_path = os.path.join(self.data_dir, self.data_type)
        if not os.path.exists(full_path):
            os.makedirs(full_path)
            if self.verbose:
                print(f"Created directory: {full_path}")
    
    def _get_file_path(self, ticker):
        """Get the file path for a given ticker (`:` in dex-prefixed names is replaced with `-`)."""
        safe = _ticker_file_safe(ticker)
        file_name = f"{safe}_{self.interval}.{self.file_type}" if self.data_type != "funding" else f"{safe}.{self.file_type}"
        return os.path.join(self.data_dir, self.data_type, file_name)
    
    def _load_existing_data(self, ticker):
        """Load existing data from file if it exists."""
        file_path = self._get_file_path(ticker)
        
        if not os.path.exists(file_path):
            return None
        
        try:
            if self.file_type == "parquet":
                df = pd.read_parquet(file_path)
            elif self.file_type == "csv":
                df = pd.read_csv(file_path)
                df['datetime'] = pd.to_datetime(df['datetime'])
            else:
                raise ValueError(f"Unsupported file type: {self.file_type}")
            
            # Ensure datetime is timezone-aware
            if df['datetime'].dt.tz is None:
                df['datetime'] = pd.to_datetime(df['datetime'], utc=True)
            
            if self.verbose:
                print(f"Loaded {len(df)} rows for {ticker} from {file_path}")
            
            return df
        except Exception as e:
            print(f"Error loading data for {ticker}: {e}")
            return None
    
    def _get_new_data(self, ticker, start_date=None):
        """Retrieve new data from Hyperliquid. To be implemented by derived classes."""
        raise NotImplementedError("Derived classes must implement _get_new_data")
    
    def _update_data(self, ticker, existing_df):
        """Update existing data with new records."""
        import datetime as dt
        
        # Calculate start date for new data
        if existing_df is not None and not existing_df.empty:
            # Get last date and subtract refresh hours
            last_date = pd.to_datetime(existing_df['datetime'].max())
            cutoff_time = last_date - dt.timedelta(hours=self.refresh_hours)
            
            # Remove data within refresh window
            rows_before = len(existing_df)
            existing_df = existing_df[existing_df['datetime'] < cutoff_time]
            rows_removed = rows_before - len(existing_df)
            
            if self.verbose and rows_removed > 0:
                print(f"  Removed {rows_removed} rows from last {self.refresh_hours} hours for refresh")
            
            start_date = cutoff_time.strftime('%Y-%m-%dT%H:%M:%SZ')
        else:
            # No existing data, get default lookback
            start_date = None
        
        # Get new data
        new_df = self._get_new_data(ticker, start_date)
        
        if new_df is None or new_df.empty:
            if self.verbose:
                print(f"  No new data retrieved for {ticker}")
            return existing_df
        
        # Combine with existing data
        if existing_df is not None and not existing_df.empty:
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            combined_df = combined_df.drop_duplicates(subset=['datetime'], keep='last')
            combined_df = combined_df.sort_values('datetime').reset_index(drop=True)
            
            if self.verbose:
                print(f"  Updated {ticker}: {len(existing_df)} -> {len(combined_df)} rows")
            
            return combined_df
        else:
            if self.verbose:
                print(f"  Downloaded {len(new_df)} rows for {ticker}")
            return new_df
    
    def _save_data(self, ticker, df):
        """Save data to file."""
        if df is None or df.empty:
            if self.verbose:
                print(f"  No data to save for {ticker}")
            return
        
        file_path = self._get_file_path(ticker)
        
        try:
            if self.file_type == "parquet":
                df.to_parquet(file_path, index=False)
            elif self.file_type == "csv":
                df.to_csv(file_path, index=False)
            else:
                raise ValueError(f"Unsupported file type: {self.file_type}")
            
            if self.verbose:
                print(f"  Saved {len(df)} rows for {ticker} to {file_path}")
        except Exception as e:
            print(f"Error saving data for {ticker}: {e}")
    
    def load_data(self):
        """
        Load data for all tickers.
        
        Returns:
            dict: Dictionary mapping tickers to their DataFrames
        """
        for ticker in self.tickers:
            if self.verbose:
                print(f"Processing {ticker}...")
            
            # Load existing data
            df = self._load_existing_data(ticker)
            
            # Update if requested
            if self.update:
                df = self._update_data(ticker, df)
                
                # Save if requested
                if self.save and df is not None:
                    self._save_data(ticker, df)
            
            # Store in data dictionary
            if df is not None:
                self.data[ticker] = df
        
        return self.data
    
    def get_data(self, ticker=None):
        """
        Get data for specific ticker(s).
        
        Args:
            ticker (str or list, optional): Ticker(s) to retrieve. If None, returns all data.
        
        Returns:
            pandas.DataFrame or dict: DataFrame if single ticker, dict if multiple
        """
        if ticker is None:
            return self.data
        elif isinstance(ticker, str):
            return self.data.get(ticker)
        elif isinstance(ticker, list):
            return {t: self.data.get(t) for t in ticker if t in self.data}
        else:
            raise ValueError("ticker must be None, str, or list")

### HyperliquidPerpManager derived class

In [ ]:

#| export
class HyperliquidPerpManager(HyperliquidDataManager):
    """
    Manager for Hyperliquid perpetual futures data.
    
    Handles reading, updating, and saving perpetual price data (OHLCV).
    Data is stored in the 'perp' subfolder with files named as {ticker}_{interval}.{file_type}
    
    Args:
        ticker (str or list, optional): Token symbol(s) to manage. If None, uses all available perp tokens.
        data_dir (str, optional): Base directory for data storage. Defaults to "../data/hyperliquid"
        interval (str, optional): Time interval for data ("1m", "5m", "15m", "1h", "4h", "1d"). 
            Defaults to "1h".
        file_type (str, optional): File format - "parquet" or "csv". Defaults to "parquet"
        update (bool, optional): If True, checks for and downloads new data. Defaults to False
        save (bool, optional): If True, saves updated data back to file. Defaults to False
        refresh_hours (int, optional): Hours of data to refresh when updating. Defaults to 24
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
        verbose (bool, optional): If True, prints progress messages. Defaults to True
    
    Examples:
        # Load existing perp data for ETH
        manager = HyperliquidPerpManager(ticker="ETH", interval="1h", info=info)
        eth_data = manager.data["ETH"]
        
        # Update and save data for multiple tokens
        manager = HyperliquidPerpManager(
            ticker=["ETH", "BTC", "SOL"],
            interval="4h",
            update=True,
            save=True,
            refresh_hours=48,
            info=info
        )
        
        # Load all available perp tokens
        manager = HyperliquidPerpManager(update=True, save=True, info=info)
    """
    
    def __init__(self, ticker=None, data_dir="../data/hyperliquid", interval="1h",
                 file_type="parquet", update=False, save=False, refresh_hours=24,
                 info=None, verbose=True):
        # Set data type before calling parent constructor
        self.data_type = "perp"
        
        # Call parent constructor
        super().__init__(ticker=ticker, data_dir=data_dir, interval=interval,
                        file_type=file_type, update=update, save=save,
                        refresh_hours=refresh_hours, info=info, verbose=verbose,data_type="perp")
        
        # Load and optionally update data for all tickers
        self._process_all_tickers()
    
    def _get_all_tokens_from_folder(self):
        """Get all tokens from the perp folder (maps file-safe names back to dex-prefixed tickers)."""
        # Load all files in the perp folder
        file_list = os.listdir(self.data_dir+'/perp')
        #perp_files = [ff for ff in file_list if f"_{self.interval}" in ff)]
        
        # Extract token names from file names
        raw_names = []
        for f in file_list:
            if self.interval in f:
                # strip _interval.ext  e.g. "xyz-XLE_1h.parquet" -> "xyz-XLE"
                stem = f.rsplit(f"_{self.interval}", 1)[0]
                raw_names.append(_ticker_from_file_safe(stem))
        
        # Return unique tokens
        return list(set(raw_names))

    def _get_all_tickers(self):
        """Get all available perpetual tokens from Hyperliquid."""
        try:
            if self.info is not None:
                tickers = hyperliquid_tokens(info=self.info)
                tickers = tickers['name'].tolist()
                if self.verbose:
                    print(f"Found {len(tickers)} perpetual tokens")
                return tickers
            else:
                # read the list of tokens from folder
                tickers = self._get_all_tokens_from_folder()
                if self.verbose:
                    print(f"Found {len(tickers)} perpetual tokens from folder")
                return tickers
        except Exception as e:
            print(f"Error getting perpetual tokens: {e}")
            return []
    
    def _get_new_data(self, ticker, start_date=None):
        """
        Retrieve new perpetual price data from Hyperliquid.
        
        Args:
            ticker (str): Token symbol
            start_date (str, optional): Start date for data retrieval. If None, uses refresh_hours.
        
        Returns:
            pandas.DataFrame: DataFrame with OHLCV data or None if error
        """
        try:
            # Calculate date range
            if start_date is None:
                end_date = None  # Will use current time
                lookback_days = self.refresh_hours / 24
            else:
                end_date = None
                lookback_days = None
            
            # Use retrieve_hyperliquid_data to get perp prices
            df = retrieve_hyperliquid_data(
                ticker=ticker,
                data_type="perp",
                start_date=start_date,
                end_date=end_date,
                lookback=lookback_days if lookback_days else 2,
                interval=self.interval,
                info=self.info
            )
            
            if df is not None and not df.empty:
                if self.verbose:
                    print(f"Retrieved {len(df)} new rows for {ticker}")
            
            return df
            
        except Exception as e:
            print(f"Error retrieving new data for {ticker}: {e}")
            return None
    
    def _update_data(self, ticker, existing_df):
        """
        Update existing perpetual data with new records.
        
        Args:
            ticker (str): Token symbol
            existing_df (pandas.DataFrame): Existing data
        
        Returns:
            pandas.DataFrame: Updated DataFrame with new data merged
        """
        import datetime as dt
        
        # Calculate start date for new data
        if existing_df is not None and not existing_df.empty:
            # Get the most recent datetime from existing data
            max_datetime = existing_df['datetime'].max()
            
            # Subtract refresh_hours to ensure overlap and catch any missing data
            start_datetime = max_datetime - pd.Timedelta(hours=self.refresh_hours)
            start_date = start_datetime.strftime('%Y-%m-%dT%H:%M:%SZ')
            
            if self.verbose:
                print(f"Updating {ticker} from {start_date}")
        else:
            # No existing data, get default lookback period
            start_date = None
            if self.verbose:
                print(f"No existing data for {ticker}, fetching initial data")
        
        # Get new data
        new_df = self._get_new_data(ticker, start_date)
        
        if new_df is None or new_df.empty:
            if self.verbose:
                print(f"No new data retrieved for {ticker}")
            return existing_df
        
        # Merge with existing data
        if existing_df is not None and not existing_df.empty:
            # Combine dataframes
            existing_df['datetime'] = existing_df['datetime'].dt.tz_localize(None)
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            # Remove duplicates based on datetime, keeping the last occurrence
            combined_df = combined_df.drop_duplicates(subset=['datetime'], keep='last')
            # Sort by datetime
            combined_df = combined_df.sort_values('datetime').reset_index(drop=True)
            if self.verbose:
                new_rows = len(combined_df) - len(existing_df)
                print(f"Added {new_rows} new rows for {ticker}")
            
            return combined_df
        else:
            # No existing data, return new data
            return new_df.sort_values('datetime').reset_index(drop=True)
    
    def _process_all_tickers(self):
        """Load and optionally update data for all tickers."""
        n = len(self.tickers)
        if self.verbose:
            print(f"Processing {n} perpetual tokens...")
        max_per_batch = 1200/100 #120 per min and 100 weighrs per hyperliquid api
        waiting_time = 2
        if n>max_per_batch:
            waiting_time = 60/max_per_batch
        if self.update:
            print(f"Warning: Processing {n} perpetual tokens exceeds the maximum number of requests per minute. Adjusting waiting time to {waiting_time} seconds.")
        for ticker in self.tickers:
            try:
                # Load existing data
                existing_df = self._load_existing_data(ticker)
                
                # Update if requested
                if self.update:
                    df = self._update_data(ticker, existing_df)
                else:
                    df = existing_df
                
                # Store in data dictionary
                if df is not None and not df.empty:
                    self.data[ticker] = df
                    
                    # Save if requested
                    if self.save and df is not None:
                        file_path = self._get_file_path(ticker)
                        save_hyperliquid_file(df, os.path.dirname(file_path), 
                                            os.path.splitext(os.path.basename(file_path))[0],
                                            type=self.file_type)
                        if self.verbose:
                            print(f"Saved {len(df)} rows for {ticker}")
                
            except Exception as e:
                print(f"Error processing {ticker}: {e}")
                continue
            if self.update:
                time.sleep(waiting_time)  # wait to avoid rate limiting
    
    def get_data(self, ticker=None):
        """
        Get data for a specific ticker.
        
        Args:
            ticker (str): Token symbol
        
        Returns:
            pandas.DataFrame: Data for the ticker or None if not available
        """
        if ticker is None:
            #a = [self.data.get(i) for i in self.tickers]
            return pd.concat(self.data.values())
        return self.data.get(ticker)
    
    def refresh_ticker(self, ticker, save=None):
        """
        Refresh data for a specific ticker.
        
        Args:
            ticker (str): Token symbol
            save (bool, optional): Override instance save setting. If None, uses instance setting.
        
        Returns:
            pandas.DataFrame: Updated data for the ticker
        """
        if ticker not in self.tickers:
            print(f"Ticker {ticker} not in managed tickers")
            return None
        
        # Load existing data
        existing_df = self._load_existing_data(ticker)
        
        # Update data
        df = self._update_data(ticker, existing_df)
        
        # Store in data dictionary
        if df is not None and not df.empty:
            self.data[ticker] = df
            
            # Save if requested
            should_save = save if save is not None else self.save
            if should_save:
                file_path = self._get_file_path(ticker)
                save_hyperliquid_file(df, os.path.dirname(file_path),
                                    os.path.splitext(os.path.basename(file_path))[0],
                                    type=self.file_type)
                if self.verbose:
                    print(f"Saved {len(df)} rows for {ticker}")
        
        return df

#### Example

In [ ]:
#| eval: false
# Load and update ETH perpetual data with 1h interval
manager = HyperliquidPerpManager(
    ticker="ETH",
    interval="1h",
    refresh_hours = 24,
    update=True,
    save=True,
    info=info
)
eth_data = manager.get_data("ETH")
print(eth_data)

Processing 1 perpetual tokens...
Loaded 26 rows for ETH from ../data/hyperliquid/perp/ETH_1h.parquet
Updating ETH from 2026-07-25T23:00:00Z
Retrieved 25 new rows for ETH
Added 0 new rows for ETH
Saved 26 rows for ETH
              datetime    open    high     low   close      volume coin
0  2026-07-25 22:00:00  1871.8  1876.0  1871.6  1875.2   2288.8254  ETH
1  2026-07-25 23:00:00  1875.2  1875.6  1872.6  1874.3   1635.3091  ETH
2  2026-07-26 00:00:00  1874.3  1880.5  1873.5  1880.0   5964.0359  ETH
3  2026-07-26 01:00:00  1880.1  1881.0  1877.4  1877.5   1205.5172  ETH
4  2026-07-26 02:00:00  1877.6  1880.5  1876.5  1880.2    925.1730  ETH
5  2026-07-26 03:00:00  1880.2  1883.4  1878.2  1881.0   2411.0634  ETH
6  2026-07-26 04:00:00  1881.4  1888.8  1879.6  1885.7   6799.9422  ETH
7  2026-07-26 05:00:00  1885.7  1886.4  1882.0  1882.3   3010.1127  ETH
8  2026-07-26 06:00:00  1882.4  1884.3  1877.6  1884.0   5390.1696  ETH
9  2026-07-26 07:00:00  1884.0  1885.5  1879.8  1880.5   1183.0

In [ ]:
#|eval: false
manager = HyperliquidPerpManager(
    update=False,
    save=False,
    verbose=False
)
df = manager.get_data()
print(df)


                      datetime    open    high     low   close    volume coin
0    2025-03-19 13:00:00+00:00  1.3062  1.3101  1.2951  1.2974   1788.20  FTT
1    2025-03-19 14:00:00+00:00  1.2989  1.3067  1.2944  1.2976   1557.00  FTT
2    2025-03-19 15:00:00+00:00  1.2988  1.3105  1.2958  1.3088   2618.00  FTT
3    2025-03-19 16:00:00+00:00  1.3070  1.3134  1.3009  1.3026   1967.90  FTT
4    2025-03-19 17:00:00+00:00  1.3029  1.3029  1.2845  1.2895   1485.00  FTT
...                        ...     ...     ...     ...     ...       ...  ...
5669 2025-11-19 18:00:00+00:00  0.2272  0.2291  0.2224  0.2238  63048.07  ACE
5670 2025-11-19 19:00:00+00:00  0.2245  0.2275  0.2238  0.2265  57691.48  ACE
5671 2025-11-19 20:00:00+00:00  0.2264  0.2313  0.2261  0.2308  78869.60  ACE
5672 2025-11-19 21:00:00+00:00  0.2312  0.2373  0.2312  0.2371  44765.71  ACE
5673 2025-11-19 22:00:00+00:00  0.2369  0.2369  0.2345  0.2360  19050.64  ACE

[854220 rows x 7 columns]


### HyperliquidSpotManager derived class

In [ ]:
#| export
class HyperliquidSpotManager(HyperliquidDataManager):
    """
    Manager for Hyperliquid spot market data.
    
    Handles reading, updating, and saving spot price data (OHLCV).
    Data is stored in the 'spot' subfolder with files named as {ticker}_{base}_{interval}.{file_type}
    
    Args:
        ticker (str or list, optional): Token symbol(s) to manage. If None, uses all available spot tokens.
        base (str, optional): Base currency for spot pairs. Defaults to "USDC".
        data_dir (str, optional): Base directory for data storage. Defaults to "../data/hyperliquid"
        interval (str, optional): Time interval for data ("1m", "5m", "15m", "1h", "4h", "1d"). 
            Defaults to "1h".
        file_type (str, optional): File format - "parquet" or "csv". Defaults to "parquet"
        update (bool, optional): If True, checks for and downloads new data. Defaults to False
        save (bool, optional): If True, saves updated data back to file. Defaults to False
        refresh_hours (int, optional): Hours of data to refresh when updating. Defaults to 24
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
        verbose (bool, optional): If True, prints progress messages. Defaults to True
    
    Examples:
        # Load existing spot data for ETH/USDC
        manager = HyperliquidSpotManager(ticker="ETH", base="USDC", interval="1h", info=info)
        eth_data = manager.data["ETH"]
        
        # Update and save data for multiple tokens
        manager = HyperliquidSpotManager(
            ticker=["ETH", "BTC", "SOL"],
            base="USDC",
            interval="4h",
            update=True,
            save=True,
            refresh_hours=48,
            info=info
        )
        
        # Load all available spot tokens
        manager = HyperliquidSpotManager(base="USDC", update=True, save=True, info=info)
    """
    
    def __init__(self, ticker=None, base="USDC", data_dir="../data/hyperliquid", interval="1h",
                 file_type="parquet", update=False, save=False, refresh_hours=24,
                 info=None, verbose=True):
        # Set data type and base before calling parent constructor
        self.data_type = "spot"
        self.base = base
        
        # Call parent constructor
        super().__init__(ticker=ticker, data_dir=data_dir, interval=interval,
                        file_type=file_type, update=update, save=save,
                        refresh_hours=refresh_hours, info=info, verbose=verbose, data_type="spot")
        
        # Load and optionally update data for all tickers
        self._process_all_tickers()
    
    def _get_file_path(self, ticker):
        """
        Get the file path for a specific ticker, including base currency.
        
        Args:
            ticker (str): Token symbol
        
        Returns:
            str: Full file path
        """
        file_name = f"{ticker}_{self.base}_{self.interval}.{self.file_type}"
        return os.path.join(self.data_dir, self.data_type, file_name)
    
    def _get_all_tickers(self):
        """Get all available spot tokens from Hyperliquid for the specified base."""
        try:
            spot_dict = {'BTC': '@142',
                        'ETH': '@151',
                        'SOL': '@156',
                        'HYPE': '@107',
                        'TRUMP': '@9',
                        'BERA': '@117',
                        'PUMP': '@20'}
            
            tickers = list(spot_dict.keys())
            if self.verbose:
                print(f"Found {len(tickers)} spot tokens for {self.base}")
            return tickers
        except Exception as e:
            print(f"Error getting spot tokens: {e}")
            return []
    
    def _get_new_data(self, ticker, start_date=None):
        """
        Retrieve new spot price data from Hyperliquid.
        
        Args:
            ticker (str): Token symbol
            start_date (str, optional): Start date for data retrieval. If None, uses refresh_hours.
        
        Returns:
            pandas.DataFrame: DataFrame with OHLCV data or None if error
        """
        try:
            # Calculate date range
            if start_date is None:
                end_date = None  # Will use current time
                lookback_days = self.refresh_hours / 24
            else:
                end_date = None
                lookback_days = None
            
            # Use retrieve_hyperliquid_data to get spot prices
            df = retrieve_hyperliquid_data(
                ticker=ticker,
                data_type="spot",
                start_date=start_date,
                end_date=end_date,
                lookback=lookback_days if lookback_days else 2,
                interval=self.interval,
                base=self.base,
                info=self.info
            )
            
            if df is not None and not df.empty:
                if self.verbose:
                    print(f"Retrieved {len(df)} new rows for {ticker}/{self.base}")
            
            return df
            
        except Exception as e:
            print(f"Error retrieving new data for {ticker}/{self.base}: {e}")
            return None
    
    def _update_data(self, ticker, existing_df):
        """
        Update existing spot data with new records.
        
        Args:
            ticker (str): Token symbol
            existing_df (pandas.DataFrame): Existing data
        
        Returns:
            pandas.DataFrame: Updated DataFrame with new data merged
        """
        import datetime as dt
        
        # Calculate start date for new data
        if existing_df is not None and not existing_df.empty:
            # Get the most recent datetime from existing data
            max_datetime = existing_df['datetime'].max()
            
            # Subtract refresh_hours to ensure overlap and catch any missing data
            start_datetime = max_datetime - pd.Timedelta(hours=self.refresh_hours)
            start_date = start_datetime.strftime('%Y-%m-%dT%H:%M:%SZ')
            
            if self.verbose:
                print(f"Updating {ticker}/{self.base} from {start_date}")
        else:
            # No existing data, get default lookback period
            start_date = None
            if self.verbose:
                print(f"No existing data for {ticker}/{self.base}, fetching initial data")
        
        # Get new data
        new_df = self._get_new_data(ticker, start_date)
        
        if new_df is None or new_df.empty:
            if self.verbose:
                print(f"No new data retrieved for {ticker}/{self.base}")
            return existing_df
        
        # Merge with existing data
        if existing_df is not None and not existing_df.empty:
            # Combine dataframes
            existing_df['datetime'] = existing_df['datetime'].dt.tz_localize(None)
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            # Remove duplicates based on datetime, keeping the last occurrence
            combined_df = combined_df.drop_duplicates(subset=['datetime'], keep='last')
            # Sort by datetime
            combined_df = combined_df.sort_values('datetime').reset_index(drop=True)
            if self.verbose:
                new_rows = len(combined_df) - len(existing_df)
                print(f"Added {new_rows} new rows for {ticker}/{self.base}")
            
            return combined_df
        else:
            # No existing data, return new data
            return new_df.sort_values('datetime').reset_index(drop=True)
    
    def _process_all_tickers(self):
        """Load and optionally update data for all tickers."""
        for ticker in self.tickers:
            try:
                # Load existing data
                existing_df = self._load_existing_data(ticker)
                
                # Update if requested
                if self.update:
                    df = self._update_data(ticker, existing_df)
                else:
                    df = existing_df
                
                # Store in data dictionary
                if df is not None and not df.empty:
                    self.data[ticker] = df
                    
                    # Save if requested
                    if self.save and df is not None:
                        file_path = self._get_file_path(ticker)
                        save_hyperliquid_file(df, os.path.dirname(file_path), 
                                            os.path.splitext(os.path.basename(file_path))[0],
                                            type=self.file_type)
                        if self.verbose:
                            print(f"Saved {len(df)} rows for {ticker}/{self.base}")
                
            except Exception as e:
                print(f"Error processing {ticker}/{self.base}: {e}")
                continue
    
    def get_data(self, ticker=None):
        """
        Get data for a specific ticker.
        
        Args:
            ticker (str): Token symbol
        
        Returns:
            pandas.DataFrame: Data for the ticker or None if not available
        """
        if ticker is None:
            return pd.concat(self.data.values())
        return self.data.get(ticker)
    
    def refresh_ticker(self, ticker, save=None):
        """
        Refresh data for a specific ticker.
        
        Args:
            ticker (str): Token symbol
            save (bool, optional): Override instance save setting. If None, uses instance setting.
        
        Returns:
            pandas.DataFrame: Updated data for the ticker
        """
        if ticker not in self.tickers:
            print(f"Ticker {ticker} not in managed tickers")
            return None
        
        # Load existing data
        existing_df = self._load_existing_data(ticker)
        
        # Update data
        df = self._update_data(ticker, existing_df)
        
        # Store in data dictionary
        if df is not None and not df.empty:
            self.data[ticker] = df
            
            # Save if requested
            should_save = save if save is not None else self.save
            if should_save:
                file_path = self._get_file_path(ticker)
                save_hyperliquid_file(df, os.path.dirname(file_path),
                                    os.path.splitext(os.path.basename(file_path))[0],
                                    type=self.file_type)
                if self.verbose:
                    print(f"Saved {len(df)} rows for {ticker}/{self.base}")
        
        return df

#### Example

In [ ]:
#| eval: false
# Example 1: Load existing spot data for a single token
print("Example 1: Load existing ETH/USDC spot data")
manager1 = HyperliquidSpotManager(
    ticker="ETH",
    base="USDC",
    interval="1h",
    update=True,
    save=True,
    info=info,
    verbose=True
)
if "ETH" in manager1.data:
    print(f"Loaded {len(manager1.data['ETH'])} records for ETH")
    print(manager1.data["ETH"])

Example 1: Load existing ETH/USDC spot data
Loaded 26 rows for ETH from ../data/hyperliquid/spot/ETH_USDC_1h.parquet
Updating ETH/USDC from 2026-07-25T23:00:00Z
Retrieved 25 new rows for ETH/USDC
Added 0 new rows for ETH/USDC
Saved 26 rows for ETH/USDC
Loaded 26 records for ETH
              datetime    open    high     low   close    volume coin
0  2026-07-25 22:00:00  1872.5  1875.3  1872.4  1875.3   16.5510  ETH
1  2026-07-25 23:00:00  1874.4  1874.8  1872.2  1873.3   59.5098  ETH
2  2026-07-26 00:00:00  1873.5  1879.4  1872.9  1879.4   52.7933  ETH
3  2026-07-26 01:00:00  1880.1  1880.1  1876.5  1876.9   54.5726  ETH
4  2026-07-26 02:00:00  1877.3  1880.2  1876.0  1879.5   45.9965  ETH
5  2026-07-26 03:00:00  1879.5  1882.5  1877.8  1880.6   46.6177  ETH
6  2026-07-26 04:00:00  1881.1  1887.2  1879.1  1884.0  600.0430  ETH
7  2026-07-26 05:00:00  1884.7  1885.4  1881.9  1881.9   30.9801  ETH
8  2026-07-26 06:00:00  1881.8  1883.8  1877.3  1883.2  214.0866  ETH
9  2026-07-26 07:00:0

In [ ]:
#| eval: false
# Example 2: Update and save data for multiple tokens
print("\nExample 2: Update and save multiple tokens")
manager2 = HyperliquidSpotManager(
    ticker=["ETH", "BTC", "SOL"],
    base="USDC",
    interval="1h",
    update=True,
    save=True,
    refresh_hours=24,
    info=info,
    verbose=True
)

# Access the data
for ticker in ["ETH", "BTC", "SOL"]:
    if ticker in manager2.data:
        print(f"\n{ticker} data shape: {manager2.data[ticker].shape}")
        print(f"Latest {ticker} price: ${manager2.data[ticker]['close'].iloc[-1]:.2f}")


Example 2: Update and save multiple tokens
Loaded 26 rows for ETH from ../data/hyperliquid/spot/ETH_USDC_1h.parquet
Updating ETH/USDC from 2026-07-25T23:00:00Z
Retrieved 25 new rows for ETH/USDC
Added 0 new rows for ETH/USDC
Saved 26 rows for ETH/USDC
Loaded 26 rows for BTC from ../data/hyperliquid/spot/BTC_USDC_1h.parquet
Updating BTC/USDC from 2026-07-25T23:00:00Z
Retrieved 25 new rows for BTC/USDC
Added 0 new rows for BTC/USDC
Saved 26 rows for BTC/USDC
Loaded 26 rows for SOL from ../data/hyperliquid/spot/SOL_USDC_1h.parquet
Updating SOL/USDC from 2026-07-25T23:00:00Z
Retrieved 25 new rows for SOL/USDC
Added 0 new rows for SOL/USDC
Saved 26 rows for SOL/USDC

ETH data shape: (26, 7)
Latest ETH price: $1959.40

BTC data shape: (26, 7)
Latest BTC price: $65393.00

SOL data shape: (26, 7)
Latest SOL price: $76.90


In [ ]:
#| eval: false
# Example 3: Load all available spot tokens with USDC base
print("\nExample 3: Load all available spot tokens")
manager3 = HyperliquidSpotManager(
    base="USDC",
    interval="4h",
    update=True,
    save=True,
    info=info,
    verbose=True
)

print(f"\nLoaded {len(manager3.data)} spot tokens")
print(f"Available tokens: {list(manager3.data.keys())}")


Example 3: Load all available spot tokens
Found 7 spot tokens for USDC
Loaded 7 rows for BTC from ../data/hyperliquid/spot/BTC_USDC_4h.parquet
Updating BTC/USDC from 2026-07-25T20:00:00Z
Retrieved 7 new rows for BTC/USDC
Added 0 new rows for BTC/USDC
Saved 7 rows for BTC/USDC
Loaded 7 rows for ETH from ../data/hyperliquid/spot/ETH_USDC_4h.parquet
Updating ETH/USDC from 2026-07-25T20:00:00Z
Retrieved 7 new rows for ETH/USDC
Added 0 new rows for ETH/USDC
Saved 7 rows for ETH/USDC
Loaded 7 rows for SOL from ../data/hyperliquid/spot/SOL_USDC_4h.parquet
Updating SOL/USDC from 2026-07-25T20:00:00Z
Retrieved 7 new rows for SOL/USDC
Added 0 new rows for SOL/USDC
Saved 7 rows for SOL/USDC
Loaded 7 rows for HYPE from ../data/hyperliquid/spot/HYPE_USDC_4h.parquet
Updating HYPE/USDC from 2026-07-25T20:00:00Z
Retrieved 7 new rows for HYPE/USDC
Added 0 new rows for HYPE/USDC
Saved 7 rows for HYPE/USDC
Loaded 1621 rows for TRUMP from ../data/hyperliquid/spot/TRUMP_USDC_4h.parquet
Updating TRUMP/USDC

In [ ]:
#| eval: false
# Example 5: Analyze spot data
print("\nExample 5: Analyze spot data")
manager5 = HyperliquidSpotManager(
    ticker=["ETH", "BTC"],
    base="USDC",
    interval="1h",
    update=True,
    info=info,
    verbose=True
)

# Calculate some basic statistics
for ticker in ["ETH", "BTC"]:
    if ticker in manager5.data:
        df = manager5.data[ticker]
        
        # Calculate 24h change
        if len(df) >= 24:
            price_24h_ago = df['close'].iloc[-24]
            current_price = df['close'].iloc[-1]
            change_24h = ((current_price - price_24h_ago) / price_24h_ago) * 100
            
            print(f"\n{ticker}/USDC:")
            print(f"  Current Price: ${current_price:.2f}")
            print(f"  24h Change: {change_24h:+.2f}%")
            print(f"  24h High: ${df['high'].iloc[-24:].max():.2f}")
            print(f"  24h Low: ${df['low'].iloc[-24:].min():.2f}")
            print(f"  24h Volume: {df['volume'].iloc[-24:].sum():.2f}")


Example 5: Analyze spot data
Loaded 26 rows for ETH from ../data/hyperliquid/spot/ETH_USDC_1h.parquet
Updating ETH/USDC from 2026-07-25T23:00:00Z
Retrieved 25 new rows for ETH/USDC
Added 0 new rows for ETH/USDC
Loaded 26 rows for BTC from ../data/hyperliquid/spot/BTC_USDC_1h.parquet
Updating BTC/USDC from 2026-07-25T23:00:00Z
Retrieved 25 new rows for BTC/USDC
Added 0 new rows for BTC/USDC

ETH/USDC:
  Current Price: $1958.30
  24h Change: +4.20%
  24h High: $1964.70
  24h Low: $1872.90
  24h Volume: 4595.75

BTC/USDC:
  Current Price: $65346.00
  24h Change: +1.42%
  24h High: $65494.00
  24h Low: $64230.00
  24h Volume: 140.47


In [ ]:
#|eval: false
mananger6 = HyperliquidSpotManager(
    base="USDC",
    interval="1h",
    update=False,
    save=False,
    verbose=True
)
mananger6.get_data()

Found 7 spot tokens for USDC
Loaded 26 rows for BTC from ../data/hyperliquid/spot/BTC_USDC_1h.parquet
Loaded 26 rows for ETH from ../data/hyperliquid/spot/ETH_USDC_1h.parquet
Loaded 26 rows for SOL from ../data/hyperliquid/spot/SOL_USDC_1h.parquet
Loaded 484 rows for HYPE from ../data/hyperliquid/spot/HYPE_USDC_1h.parquet
Loaded 36 rows for TRUMP from ../data/hyperliquid/spot/TRUMP_USDC_1h.parquet
Loaded 25 rows for BERA from ../data/hyperliquid/spot/BERA_USDC_1h.parquet
Loaded 480 rows for PUMP from ../data/hyperliquid/spot/PUMP_USDC_1h.parquet


,datetime,open,high,low,close,volume,coin
0,2026-07-25 22:00:00+00:00,64303.000000,64356.000000,64290.000000,64350.000000,2.847586e+01,BTC
1,2026-07-25 23:00:00+00:00,64351.000000,64363.000000,64300.000000,64305.000000,1.306880e+00,BTC
2,2026-07-26 00:00:00+00:00,64304.000000,64433.000000,64300.000000,64433.000000,5.077590e+00,BTC
3,2026-07-26 01:00:00+00:00,64437.000000,64481.000000,64399.000000,64402.000000,3.229600e+00,BTC
4,2026-07-26 02:00:00+00:00,64407.000000,64470.000000,64373.000000,64468.000000,1.359620e+00,BTC
...,...,...,...,...,...,...,...
475,2025-11-19 13:00:00+00:00,0.000148,0.000148,0.000148,0.000148,0.000000e+00,PUMP
476,2025-11-19 14:00:00+00:00,0.000148,0.000151,0.000148,0.000151,1.585506e+06,PUMP
477,2025-11-19 15:00:00+00:00,0.000151,0.000151,0.000151,0.000151,0.000000e+00,PUMP
478,2025-11-19 16:00:00+00:00,0.000151,0.000151,0.000151,0.000151,0.000000e+00,PUMP


### HyperliquidFundingManager derived class

In [ ]:
#| export
class HyperliquidFundingManager(HyperliquidDataManager):
    """
    Manager for Hyperliquid funding rate data.
    
    Handles reading, updating, and saving funding rate data.
    Data is stored in the 'funding' subfolder with files named as {ticker}.{file_type}
    
    Args:
        ticker (str or list, optional): Token symbol(s) to manage. If None, uses all available tokens.
        data_dir (str, optional): Base directory for data storage. Defaults to "../data/hyperliquid"
        file_type (str, optional): File format - "parquet" or "csv". Defaults to "parquet"
        update (bool, optional): If True, checks for and downloads new data. Defaults to False
        save (bool, optional): If True, saves updated data back to file. Defaults to False
        refresh_hours (int, optional): Hours of data to refresh when updating. Defaults to 24
        round_to_hour (bool, optional): If True, rounds datetime to nearest hour. Defaults to True
        info (Info, optional): Hyperliquid Info client. If None, creates a new one.
        verbose (bool, optional): If True, prints progress messages. Defaults to True
    
    Examples:
        # Load existing funding data for ETH
        manager = HyperliquidFundingManager(ticker="ETH", info=info)
        eth_data = manager.data["ETH"]
        
        # Update and save data for multiple tokens
        manager = HyperliquidFundingManager(
            ticker=["ETH", "BTC", "SOL"],
            update=True,
            save=True,
            refresh_hours=48,
            info=info
        )
        
        # Load all available tokens
        manager = HyperliquidFundingManager(update=True, save=True, info=info)
    """
    
    def __init__(self, ticker=None, data_dir="../data/hyperliquid", 
                 file_type="parquet", update=False, save=False, refresh_hours=24,
                 round_to_hour=True, info=None, verbose=True):
        # Set data type and round_to_hour before calling parent constructor
        self.data_type = "funding"
        self.round_to_hour = round_to_hour
        
        # Call parent constructor (interval not used for funding data, but required by parent)
        super().__init__(ticker=ticker, data_dir=data_dir, interval="1h",
                        file_type=file_type, update=update, save=save,
                        refresh_hours=refresh_hours, info=info, verbose=verbose,
                        data_type="funding")
        
        # Load and optionally update data for all tickers
        self._process_all_tickers()
    
    def _get_all_tokens_from_folder(self):
        """Get all tokens from the funding folder (maps file-safe names back to dex-prefixed tickers)."""
        # Load all files in the funding folder
        file_list = os.listdir(self.data_dir+'/funding')
        
        # Extract token names from file names
        tickers = [_ticker_from_file_safe(f.split(".")[0]) for f in file_list]
        
        # Return unique tokens
        return list(set(tickers))

    def _get_all_tickers(self):
        """Get all available tokens from Hyperliquid."""
        try:
            if self.info is None:
                # read the list of tokens from folder
                tickers = self._get_all_tokens_from_folder()
                if self.verbose:
                    print(f"Found {len(tickers)} perpetual tokens from folder")
                return tickers
            tickers = hyperliquid_tokens(info=self.info)
            tickers = tickers['name'].tolist()
            if self.verbose:
                print(f"Found {len(tickers)} tokens")
            return tickers
        except Exception as e:
            print(f"Error getting tokens: {e}")
            return []
    
    def _get_file_path(self, ticker):
        """Get the file path for a given ticker (funding data doesn't use interval in filename)."""
        safe = _ticker_file_safe(ticker)
        file_name = f"{safe}.{self.file_type}"
        return os.path.join(self.data_dir, self.data_type, file_name)
    
    def _get_new_data(self, ticker, start_date=None):
        """
        Retrieve new funding rate data from Hyperliquid.
        
        Args:
            ticker (str): Token symbol
            start_date (str, optional): Start date for data retrieval. If None, uses refresh_hours.
        
        Returns:
            pandas.DataFrame: DataFrame with funding rate data or None if error
        """
        try:
            # Calculate date range
            if start_date is None:
                end_date = None  # Will use current time
                lookback_days = self.refresh_hours / 24
            else:
                end_date = None
                lookback_days = None
            
            # Use retrieve_hyperliquid_data to get funding rates
            df = retrieve_hyperliquid_data(
                ticker=ticker,
                data_type="funding",
                start_date=start_date,
                end_date=end_date,
                lookback=lookback_days if lookback_days else 2,
                round_to_hour=self.round_to_hour,
                info=self.info
            )
            
            if df is not None and not df.empty:
                if self.verbose:
                    print(f"Retrieved {len(df)} new rows for {ticker}")
            
            return df
            
        except Exception as e:
            print(f"Error retrieving new data for {ticker}: {e}")
            return None
    
    def _update_data(self, ticker, existing_df):
        """
        Update existing funding rate data with new records.
        
        Args:
            ticker (str): Token symbol
            existing_df (pandas.DataFrame): Existing data
        
        Returns:
            pandas.DataFrame: Updated DataFrame with new data merged
        """
        import datetime as dt
        
        # Calculate start date for new data
        if existing_df is not None and not existing_df.empty:
            # Get the most recent datetime from existing data
            max_datetime = existing_df['datetime'].max()
            
            # Subtract refresh_hours to ensure overlap and catch any missing data
            start_datetime = max_datetime - pd.Timedelta(hours=self.refresh_hours)
            start_date = start_datetime.strftime('%Y-%m-%dT%H:%M:%SZ')
            
            if self.verbose:
                print(f"Updating {ticker} from {start_date}")
        else:
            # No existing data, get default lookback period
            start_date = None
            if self.verbose:
                print(f"No existing data for {ticker}, fetching initial data")
        
        # Get new data
        new_df = self._get_new_data(ticker, start_date)
        
        if new_df is None or new_df.empty:
            if self.verbose:
                print(f"No new data retrieved for {ticker}")
            return existing_df
        
        # Merge with existing data
        if existing_df is not None and not existing_df.empty:
            # Combine dataframes
            existing_df['datetime'] = existing_df['datetime'].dt.tz_localize(None)
            combined_df = pd.concat([existing_df, new_df], ignore_index=True)
            # Remove duplicates based on datetime, keeping the last occurrence
            combined_df = combined_df.drop_duplicates(subset=['datetime'], keep='last')
            # Sort by datetime
            combined_df = combined_df.sort_values('datetime').reset_index(drop=True)
            if self.verbose:
                new_rows = len(combined_df) - len(existing_df)
                print(f"Added {new_rows} new rows for {ticker}")
            
            return combined_df
        else:
            # No existing data, return new data
            return new_df.sort_values('datetime').reset_index(drop=True)
    
    def _process_all_tickers(self):
        """Load and optionally update data for all tickers."""
        n = len(self.tickers)
        max_per_batch = 1200/100 #120 per min and 100 weighrs per hyperliquid api
        waiting_time = 2
        if n>max_per_batch:
            waiting_time = 60/max_per_batch
        if self.update:
            print(f"Warning: Processing {n} perpetual tokens exceeds the maximum number of requests per minute. Adjusting waiting time to {waiting_time} seconds.")
        for ticker in self.tickers:
            try:
                # Load existing data
                existing_df = self._load_existing_data(ticker)
                
                # Update if requested
                if self.update:
                    df = self._update_data(ticker, existing_df)
                else:
                    df = existing_df
                
                # Store in data dictionary
                if df is not None and not df.empty:
                    self.data[ticker] = df
                    
                    # Save if requested
                    if self.save and df is not None:
                        file_path = self._get_file_path(ticker)
                        save_hyperliquid_file(df, os.path.dirname(file_path), 
                                            os.path.splitext(os.path.basename(file_path))[0],
                                            type=self.file_type)
                        if self.verbose:
                            print(f"Saved {len(df)} rows for {ticker}")
                
            except Exception as e:
                print(f"Error processing {ticker}: {e}")
                continue
            if self.update:
                time.sleep(waiting_time)    
    def get_data(self, ticker=None):
        """
        Get data for a specific ticker.
        
        Args:
            ticker (str): Token symbol
        
        Returns:
            pandas.DataFrame: Data for the ticker or None if not available
        """
        if ticker is None:
            return pd.concat(self.data.values())
        return self.data.get(ticker)
    
    def refresh_ticker(self, ticker, save=None):
        """
        Refresh data for a specific ticker.
        
        Args:
            ticker (str): Token symbol
            save (bool, optional): Override instance save setting. If None, uses instance setting.
        
        Returns:
            pandas.DataFrame: Updated data for the ticker
        """
        if ticker not in self.tickers:
            print(f"Ticker {ticker} not in managed tickers")
            return None
        
        # Load existing data
        existing_df = self._load_existing_data(ticker)
        
        # Update data
        df = self._update_data(ticker, existing_df)
        
        # Store in data dictionary
        if df is not None and not df.empty:
            self.data[ticker] = df
            
            # Save if requested
            should_save = save if save is not None else self.save
            if should_save:
                file_path = self._get_file_path(ticker)
                save_hyperliquid_file(df, os.path.dirname(file_path),
                                    os.path.splitext(os.path.basename(file_path))[0],
                                    type=self.file_type)
                if self.verbose:
                    print(f"Saved {len(df)} rows for {ticker}")
        
        return df

#### Example

In [ ]:
#|eval:false
manager = HyperliquidFundingManager(ticker="ETH")
eth_data = manager.data["ETH"]
print(eth_data)

Loaded 738 rows for ETH from ../data/hyperliquid/funding/ETH.parquet
                     datetime  funding_rate   premium coin  fund_calc
0   2025-10-03 21:00:00+00:00      0.000013  0.000185  ETH   0.000013
1   2025-10-03 22:00:00+00:00      0.000013  0.000111  ETH   0.000013
2   2025-10-03 23:00:00+00:00      0.000013  0.000141  ETH   0.000013
3   2025-10-04 00:00:00+00:00      0.000013  0.000113  ETH   0.000013
4   2025-10-04 01:00:00+00:00      0.000013  0.000081  ETH   0.000013
..                        ...           ...       ...  ...        ...
733 2025-11-19 07:00:00+00:00      0.000013  0.000175  ETH   0.000013
734 2025-11-19 08:00:00+00:00      0.000013  0.000281  ETH   0.000013
735 2025-11-19 09:00:00+00:00      0.000013  0.000277  ETH   0.000013
736 2025-11-19 10:00:00+00:00      0.000013  0.000256  ETH   0.000013
737 2025-11-19 11:00:00+00:00      0.000013  0.000295  ETH   0.000013

[738 rows x 5 columns]


In [ ]:
#|eval:false
manager = HyperliquidFundingManager(update=False, save=False,verbose=False)
manager.get_data()

,datetime,funding_rate,premium,coin,fund_calc
0,2025-10-03 21:00:00+00:00,-0.000013,-0.000606,FTT,-0.000013
1,2025-10-03 22:00:00+00:00,0.000013,-0.000313,FTT,0.000013
2,2025-10-03 23:00:00+00:00,0.000013,-0.000128,FTT,0.000013
3,2025-10-04 00:00:00+00:00,0.000013,-0.000041,FTT,0.000013
4,2025-10-04 01:00:00+00:00,0.000013,-0.000069,FTT,0.000013
...,...,...,...,...,...
733,2025-11-19 07:00:00+00:00,0.000013,-0.000111,ACE,0.000013
734,2025-11-19 08:00:00+00:00,0.000013,-0.000264,ACE,0.000013
735,2025-11-19 09:00:00+00:00,0.000013,-0.000378,ACE,0.000013
736,2025-11-19 10:00:00+00:00,0.000013,-0.000113,ACE,0.000013


## More examples

Load package:

In [ ]:
#|eval:false
from token_data.hyperliquid import *

Load all available funding rates already stored in the data directory. It returns a pandas DataFrame with columns 'datetime' in UTC and 'funding_rate' as the funding rate used within the Hyperliquid system to find the USDC paid or received for a given perpetual position.

The other coluns are "premium" and "fund_calc". The "premium" column represents the premium used in the funding rate calculation and the "fund_calc" is currently very experimental and tries to replicated the funding rate calculation in the Hyperliquid system. Use "funding_rate" column in simulations or P&L calculations.

In [ ]:
#|eval:false
manager = HyperliquidFundingManager(update=False, save=False, verbose=False)
manager.get_data()

,datetime,funding_rate,premium,coin,fund_calc
0,2025-10-03 21:00:00+00:00,-0.000013,-0.000606,FTT,-0.000013
1,2025-10-03 22:00:00+00:00,0.000013,-0.000313,FTT,0.000013
2,2025-10-03 23:00:00+00:00,0.000013,-0.000128,FTT,0.000013
3,2025-10-04 00:00:00+00:00,0.000013,-0.000041,FTT,0.000013
4,2025-10-04 01:00:00+00:00,0.000013,-0.000069,FTT,0.000013
...,...,...,...,...,...
733,2025-11-19 07:00:00+00:00,0.000013,-0.000111,ACE,0.000013
734,2025-11-19 08:00:00+00:00,0.000013,-0.000264,ACE,0.000013
735,2025-11-19 09:00:00+00:00,0.000013,-0.000378,ACE,0.000013
736,2025-11-19 10:00:00+00:00,0.000013,-0.000113,ACE,0.000013


Load all available spot prices already stored in the data directory. It returns a pandas DataFrame with columns 'datetime' in UTC and 'open', 'high', 'low', 'close', 'volume' and 'coin'. Important to note that the list of spots is limited and was hard coded. In the future this can be changed, therefore, the list is limited to some of the most important tokens.

In [ ]:
#|eval:false
manager = HyperliquidSpotManager(update=False, save=False, verbose=False)
manager.get_data()

,datetime,open,high,low,close,volume,coin
0,2026-07-25 22:00:00+00:00,64303.000000,64356.000000,64290.000000,64350.000000,2.847586e+01,BTC
1,2026-07-25 23:00:00+00:00,64351.000000,64363.000000,64300.000000,64305.000000,1.306880e+00,BTC
2,2026-07-26 00:00:00+00:00,64304.000000,64433.000000,64300.000000,64433.000000,5.077590e+00,BTC
3,2026-07-26 01:00:00+00:00,64437.000000,64481.000000,64399.000000,64402.000000,3.229600e+00,BTC
4,2026-07-26 02:00:00+00:00,64407.000000,64470.000000,64373.000000,64468.000000,1.359620e+00,BTC
...,...,...,...,...,...,...,...
475,2025-11-19 13:00:00+00:00,0.000148,0.000148,0.000148,0.000148,0.000000e+00,PUMP
476,2025-11-19 14:00:00+00:00,0.000148,0.000151,0.000148,0.000151,1.585506e+06,PUMP
477,2025-11-19 15:00:00+00:00,0.000151,0.000151,0.000151,0.000151,0.000000e+00,PUMP
478,2025-11-19 16:00:00+00:00,0.000151,0.000151,0.000151,0.000151,0.000000e+00,PUMP


Next example loads perpetual prices already stored in the data directory on an hourly frequency (default but can be changed). It returns a pandas DataFrame with columns 'datetime' in UTC and 'open', 'high', 'low', 'close', 'volume' and 'coin'.

In [ ]:
#|eval:false
manager = HyperliquidPerpManager(update=False, save=False,verbose=False)
manager.get_data()

,datetime,open,high,low,close,volume,coin
0,2025-03-19 13:00:00+00:00,1.3062,1.3101,1.2951,1.2974,1788.20,FTT
1,2025-03-19 14:00:00+00:00,1.2989,1.3067,1.2944,1.2976,1557.00,FTT
2,2025-03-19 15:00:00+00:00,1.2988,1.3105,1.2958,1.3088,2618.00,FTT
3,2025-03-19 16:00:00+00:00,1.3070,1.3134,1.3009,1.3026,1967.90,FTT
4,2025-03-19 17:00:00+00:00,1.3029,1.3029,1.2845,1.2895,1485.00,FTT
...,...,...,...,...,...,...,...
5669,2025-11-19 18:00:00+00:00,0.2272,0.2291,0.2224,0.2238,63048.07,ACE
5670,2025-11-19 19:00:00+00:00,0.2245,0.2275,0.2238,0.2265,57691.48,ACE
5671,2025-11-19 20:00:00+00:00,0.2264,0.2313,0.2261,0.2308,78869.60,ACE
5672,2025-11-19 21:00:00+00:00,0.2312,0.2373,0.2312,0.2371,44765.71,ACE


Next example loads only "ETH" perpetual prices already stored in the data directory on an hourly frequency (default but can be changed). It returns a pandas DataFrame with columns 'datetime' in UTC.

In [ ]:
#|eval:false
manager = HyperliquidPerpManager(ticker='ETH',update=False, save=False,verbose=True)
manager.get_data()

Processing 1 perpetual tokens...
Loaded 26 rows for ETH from ../data/hyperliquid/perp/ETH_1h.parquet


,datetime,open,high,low,close,volume,coin
0,2026-07-25 22:00:00+00:00,1871.8,1876.0,1871.6,1875.2,2288.8254,ETH
1,2026-07-25 23:00:00+00:00,1875.2,1875.6,1872.6,1874.3,1635.3091,ETH
2,2026-07-26 00:00:00+00:00,1874.3,1880.5,1873.5,1880.0,5964.0359,ETH
3,2026-07-26 01:00:00+00:00,1880.1,1881.0,1877.4,1877.5,1205.5172,ETH
4,2026-07-26 02:00:00+00:00,1877.6,1880.5,1876.5,1880.2,925.1730,ETH
5,2026-07-26 03:00:00+00:00,1880.2,1883.4,1878.2,1881.0,2411.0634,ETH
6,2026-07-26 04:00:00+00:00,1881.4,1888.8,1879.6,1885.7,6799.9422,ETH
7,2026-07-26 05:00:00+00:00,1885.7,1886.4,1882.0,1882.3,3010.1127,ETH
8,2026-07-26 06:00:00+00:00,1882.4,1884.3,1877.6,1884.0,5390.1696,ETH
9,2026-07-26 07:00:00+00:00,1884.0,1885.5,1879.8,1880.5,1183.0028,ETH


If you want to make sure you are featching the most recent data, use the `update=True` parameter. In that case one needs to pass the `info` parameter with your API key to retrieve real-time data which is setup beforehand. The next example demonstrates how to use this:

In [ ]:
#|eval:false
address, info, exchange = setup(base_url=constants.MAINNET_API_URL, skip_ws=True)
manager = HyperliquidPerpManager(ticker='ETH',update=True, save=False, verbose=True,info=info)
manager.get_data()

Processing 1 perpetual tokens...
Loaded 26 rows for ETH from ../data/hyperliquid/perp/ETH_1h.parquet
Updating ETH from 2026-07-25T23:00:00Z
Retrieved 25 new rows for ETH
Added 0 new rows for ETH


,datetime,open,high,low,close,volume,coin
0,2026-07-25 22:00:00,1871.8,1876.0,1871.6,1875.2,2288.8254,ETH
1,2026-07-25 23:00:00,1875.2,1875.6,1872.6,1874.3,1635.3091,ETH
2,2026-07-26 00:00:00,1874.3,1880.5,1873.5,1880.0,5964.0359,ETH
3,2026-07-26 01:00:00,1880.1,1881.0,1877.4,1877.5,1205.5172,ETH
4,2026-07-26 02:00:00,1877.6,1880.5,1876.5,1880.2,925.1730,ETH
5,2026-07-26 03:00:00,1880.2,1883.4,1878.2,1881.0,2411.0634,ETH
6,2026-07-26 04:00:00,1881.4,1888.8,1879.6,1885.7,6799.9422,ETH
7,2026-07-26 05:00:00,1885.7,1886.4,1882.0,1882.3,3010.1127,ETH
8,2026-07-26 06:00:00,1882.4,1884.3,1877.6,1884.0,5390.1696,ETH
9,2026-07-26 07:00:00,1884.0,1885.5,1879.8,1880.5,1183.0028,ETH


The same goes for the other data items. Such as spot prices and funding rates. Next example shows the spot prices for "ETH" on an hourly frequency and updates it:

In [ ]:
#|eval:false
manager = HyperliquidSpotManager(ticker='ETH',update=True, save=False, verbose=True,info=info)
manager.get_data()

Loaded 26 rows for ETH from ../data/hyperliquid/spot/ETH_USDC_1h.parquet
Updating ETH/USDC from 2026-07-25T23:00:00Z
Retrieved 25 new rows for ETH/USDC
Added 0 new rows for ETH/USDC


,datetime,open,high,low,close,volume,coin
0,2026-07-25 22:00:00,1872.5,1875.3,1872.4,1875.3,16.5510,ETH
1,2026-07-25 23:00:00,1874.4,1874.8,1872.2,1873.3,59.5098,ETH
2,2026-07-26 00:00:00,1873.5,1879.4,1872.9,1879.4,52.7933,ETH
3,2026-07-26 01:00:00,1880.1,1880.1,1876.5,1876.9,54.5726,ETH
4,2026-07-26 02:00:00,1877.3,1880.2,1876.0,1879.5,45.9965,ETH
5,2026-07-26 03:00:00,1879.5,1882.5,1877.8,1880.6,46.6177,ETH
6,2026-07-26 04:00:00,1881.1,1887.2,1879.1,1884.0,600.0430,ETH
7,2026-07-26 05:00:00,1884.7,1885.4,1881.9,1881.9,30.9801,ETH
8,2026-07-26 06:00:00,1881.8,1883.8,1877.3,1883.2,214.0866,ETH
9,2026-07-26 07:00:00,1883.9,1884.7,1879.2,1879.9,200.7441,ETH


If you want to update the funding rate, do the following:

In [ ]:
#|eval:false
manager = HyperliquidFundingManager(ticker='ETH',update=True, save=False, verbose=True,info=info)
manager.get_data()

Loaded 738 rows for ETH from ../data/hyperliquid/funding/ETH.parquet
Updating ETH from 2025-11-18T11:00:00Z
Retrieved 500 new rows for ETH
Added 475 new rows for ETH


,datetime,funding_rate,premium,coin,fund_calc
0,2025-10-03 21:00:00,0.000013,0.000185,ETH,0.000013
1,2025-10-03 22:00:00,0.000013,0.000111,ETH,0.000013
2,2025-10-03 23:00:00,0.000013,0.000141,ETH,0.000013
3,2025-10-04 00:00:00,0.000013,0.000113,ETH,0.000013
4,2025-10-04 01:00:00,0.000013,0.000081,ETH,0.000013
...,...,...,...,...,...
1208,2025-12-09 02:00:00,0.000013,0.000120,ETH,0.000013
1209,2025-12-09 03:00:00,0.000013,0.000032,ETH,0.000013
1210,2025-12-09 04:00:00,0.000013,0.000102,ETH,0.000013
1211,2025-12-09 05:00:00,0.000013,0.000196,ETH,0.000013


## RWA / HIP-3 examples
The cells below demonstrate the auto-resolution of RWA tickers (e.g. `XLE` → `xyz:XLE`).

In [ ]:
#| eval: false
# Resolve ticker
print("resolve_hyperliquid_ticker('ETH') ->", resolve_hyperliquid_ticker("ETH"))
print("resolve_hyperliquid_ticker('XLE') ->", resolve_hyperliquid_ticker("XLE"))
print("resolve_hyperliquid_ticker('xyz:XLE') ->", resolve_hyperliquid_ticker("xyz:XLE"))

resolve_hyperliquid_ticker('ETH') -> ETH
resolve_hyperliquid_ticker('XLE') -> xyz:XLE
resolve_hyperliquid_ticker('xyz:XLE') -> xyz:XLE


In [ ]:
#| eval: false
# RWA perp price
xle_perp = retrieve_hyperliquid_perp_price(coin="XLE", interval="1h", info=info)
print("XLE perp price:")
print(xle_perp.head())
print(f"Columns: {list(xle_perp.columns)}")
print(f"coin value: {xle_perp['coin'].iloc[0]}")

XLE perp price:
             datetime    open    high     low   close  volume coin
0 2026-07-24 23:00:00  59.789  59.793  59.789  59.793    3.59  XLE
1 2026-07-25 00:00:00  59.960  59.960  59.559  59.642   46.08  XLE
2 2026-07-25 01:00:00  59.601  59.712  59.453  59.671  334.74  XLE
3 2026-07-25 02:00:00  59.742  59.835  59.692  59.834   68.50  XLE
4 2026-07-25 03:00:00  59.859  59.859  59.859  59.859    1.67  XLE
Columns: ['datetime', 'open', 'high', 'low', 'close', 'volume', 'coin']
coin value: XLE


In [ ]:
#| eval: false
# RWA funding history
xle_funding = retrieve_hyperliquid_funding_history(coin="XLE", info=info)
print("XLE funding history:")
print(xle_funding.head())

XLE funding history:
                 datetime  funding_rate   premium coin  fund_calc
0 2026-07-25 00:00:00.066      0.000006  0.000206  XLE   0.000006
1 2026-07-25 01:00:00.036      0.000006 -0.000030  XLE   0.000006
2 2026-07-25 02:00:00.050     -0.000025 -0.000703  XLE  -0.000025
3 2026-07-25 03:00:00.016      0.000027  0.000731  XLE   0.000027
4 2026-07-25 04:00:00.036      0.000058  0.001233  XLE   0.000058


In [ ]:
#| eval: false
# RWA mid price
xle_mid = hyperliquid_mids(coin="XLE", info=info)
print(f"XLE mid price: {xle_mid}")

XLE mid price: 58.78


In [ ]:
#| eval: false
# RWA L2 snapshot
xle_l2 = retrieve_hyperliquid_l2_snapshot(coin="XLE", info=info)
print("XLE L2 snapshot:")
print(xle_l2.head())

XLE L2 snapshot:
                 datetime side   price    size  num_orders
0 2026-07-26 23:17:56.011  bid  58.735   17.03           1
1 2026-07-26 23:17:56.011  bid  58.734   54.97           1
2 2026-07-26 23:17:56.011  bid  58.733   17.03           1
3 2026-07-26 23:17:56.011  bid  58.732  855.76           2
4 2026-07-26 23:17:56.011  bid  58.731   39.28           1


In [ ]:
#| eval: false
# List HIP-3 dex tokens
rwa_tokens = hyperliquid_tokens(info=info, dex="xyz")
print("xyz dex tokens:")
print(rwa_tokens.head(10))

xyz dex tokens:
    szDecimals        name  maxLeverage  marginTableId growthMode  \
0            4  xyz:XYZ100           30             30    enabled   
1            3    xyz:TSLA           20             20    enabled   
2            3    xyz:NVDA           20             20    enabled   
3            4    xyz:GOLD           25             25        NaN   
8            3    xyz:META           20             20    enabled   
9            3    xyz:AAPL           20             20    enabled   
10           3    xyz:MSFT           20             20    enabled   
12           3   xyz:GOOGL           20             20    enabled   
13           3    xyz:AMZN           20             20    enabled   
15           3      xyz:MU           10             10    enabled   

         lastGrowthModeChangeTime  onlyIsolated marginMode  isDelisted  
0   2025-11-23T17:37:10.033211662         False        NaN       False  
1   2025-11-23T17:37:10.033211662         False        NaN       False  
2   2

In [ ]:
#| eval: false
# Regression check: ETH still works
eth_perp = retrieve_hyperliquid_perp_price(coin="ETH", interval="1h", info=info)
eth_funding = retrieve_hyperliquid_funding_history(coin="ETH", info=info)
print(f"ETH perp rows: {len(eth_perp)}, columns: {list(eth_perp.columns)}")
print(f"ETH funding rows: {len(eth_funding)}, columns: {list(eth_funding.columns)}")
print("ETH perp head:")
print(eth_perp.head())

ETH perp rows: 49, columns: ['datetime', 'open', 'high', 'low', 'close', 'volume', 'coin']
ETH funding rows: 48, columns: ['datetime', 'funding_rate', 'premium', 'coin', 'fund_calc']
ETH perp head:
             datetime    open    high     low   close     volume coin
0 2026-07-24 23:00:00  1858.8  1861.1  1858.1  1861.0  2792.8168  ETH
1 2026-07-25 00:00:00  1860.7  1863.7  1856.3  1859.3  4137.4891  ETH
2 2026-07-25 01:00:00  1859.3  1862.6  1857.4  1859.3  4557.0876  ETH
3 2026-07-25 02:00:00  1859.3  1860.7  1857.3  1858.3  4404.2636  ETH
4 2026-07-25 03:00:00  1858.3  1859.5  1854.8  1858.3  4298.4131  ETH


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()